# 30. MMPDB 통합 - beta-keto/anhydride 재검증 + 통계적 유의성

## 이번 노트북에서 할 것
- MMPDB(공식 Matched Molecular Pair 도구) 설치
- Tox21 데이터로 fragment/index/loadprops 파이프라인 구축
- beta-keto/anhydride 등 오늘 "데이터형 불가"로 결론 내린 규칙들을
  MMPDB의 --min-pairs, --min-radius 옵션으로 재검증
- rule_environment_statistics의 PAIRED_T, P_VALUE를 활용해 우리 규칙의
  통계적 신뢰도 재확인

## 간략한 정리 (29까지)
- 라이브러리 30개 규칙(thiol_1 세분화로 실질 31개), 11가지 편집 방식
- valid set 커버리지 31.5%, 단일문제분자 성공률(최소1단계) 84%
- 4개 endpoint 교차검증: Ames(p<0.0001), Tox21(p=0.032, 유의해짐),
  DILI(p=0.0018, 신규), hERG(p=0.363, 미유의)
- 오늘(29번) 발견/수정한 버그 4건: replace_ring 조각화(다중치환기 미고려),
  cleave_bond 회귀(fragmentation guard 과잉적용), iterative_fix_loop
  재시도 로직 결여(첫 시도 실패시 즉시 stuck), Sulfonic_acid_2/thiol_1
  SMARTS 협소함(설페이트에스터/티오카르복실산염 혼동)
- 순서 의존성 실험(20개 표본): 최종상태 다른 5건 중 LLM이 4건 우위
- AI신약연구원(aidd.kr) MMPDB 활용가이드 확인 - 공식 통계검정/필터링
  기능이 이미 있어 재활용 가치 있음
- test set은 여전히 미사용

## 다음에 해야 할 것
- 나머지 stuck 케이스(thioester, catechol 재현안됨, Three-membered_
  heterocycle) 원인 조사
- phthalimide, hydroxamic_acid 문헌형은 학생이 논문 확인 후 진행
- 학생 승인 시 test set 1회 최종 검증

In [1]:
# 셀 1
!pip install mmpdb -q
!pip install rdkit -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.9/47.9 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.3/245.3 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 31.5 MB/s eta 0:00:00


In [2]:
# 셀 2
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')

!git clone https://{token}@github.com/Dec32th/laidd-2026.git
%cd /content/laidd-2026
!pwd

Cloning into 'laidd-2026'...
remote: Enumerating objects: 361, done.
remote: Counting objects: 100% (101/101), done.
remote: Compressing objects: 100% (67/67), done.
remote: Total 361 (delta 53), reused 79 (delta 34), pack-reused 260 (from 1)
Receiving objects: 100% (361/361), 977.06 KiB | 21.24 MiB/s, done.
Resolving deltas: 100% (190/190), done.
/content/laidd-2026
/content/laidd-2026


In [3]:
# 셀 3
!git config --global user.email "hyekyeong.w@gmail.com"
!git config --global user.name "Dec32th"

In [4]:
# 셀 4
!mmpdb --help

Usage: mmpdb [OPTIONS] COMMAND [ARGS]...

  Matched-molecular pair database loader

Options:
  -q, --quiet  Do not show progress or status information
  --version    Show the version and exit.
  --help       Show this message and exit.

Commands:

 Matched molecular pair generation commands (see 'help-analysis'):
  fragment       Fragment SMILES file structures on rotatable bonds
  smifrag        Fragment a single SMILES string
  index          Index fragments and find matched molecular pairs
  predict        Predict the effect of a structural transformation
  transform      Transform a structure
  rgroup2smarts  Convert an R-group file into a SMARTS which matches all of
                 the SMILES
  generate       Apply database transforms to a molecule

 Distributed generation commands (see 'help-distributed'):
  smi_split         Split the SMILES file 'FILE' into smaller files
  fragdb_constants  List constants fragdb DATABASEs and their frequencies
  fragdb_partition  Partition fra

In [5]:
import importlib
import src.tools.replacement_library
import src.tools.molecule_editor
import src.tools.atom_editor
import src.tools.toxicophore_detector
from src.tools.data_prep import load_tox21_clean
from src.tools.toxicophore_detector import detect_toxicophores
from src.tools.replacement_library import get_replacement_candidates

data = load_tox21_clean(random_state=7)

# MMPDB는 "SMILES ID" 형식의 .smi 파일을 요구함 (한 줄에 하나씩, 공백으로 구분)
with open("tox21_valid.smi", "w") as f:
    for i, smi in enumerate(data['smiles_valid']):
        f.write(f"{smi} mol_{i}\n")

print(f"저장 완료: {len(data['smiles_valid'])}개 분자")
!head -5 tox21_valid.smi

[16:02:55] WARNING: not removing hydrogen atom without neighbors
[16:02:55] Explicit valence for atom # 8 Al, 6, is greater than permitted
[16:02:56] Explicit valence for atom # 3 Al, 6, is greater than permitted
[16:02:56] Explicit valence for atom # 4 Al, 6, is greater than permitted
[16:02:57] Explicit valence for atom # 4 Al, 6, is greater than permitted
[16:02:57] Explicit valence for atom # 9 Al, 6, is greater than permitted
[16:02:57] Explicit valence for atom # 5 Al, 6, is greater than permitted
[16:02:58] Explicit valence for atom # 16 Al, 6, is greater than permitted
[16:02:58] Explicit valence for atom # 20 Al, 6, is greater than permitted


전체: 7831개, 파싱 성공: 7823개, 파싱 실패(제외): 8개


[16:02:59] WARNING: not removing hydrogen atom without neighbors


저장 완료: 1173개 분자
Cc1cc(NS(=O)(=O)c2ccc(N)cc2)nc(C)n1 mol_0
CCCOC(=O)c1ccc(C(=O)OCCC)nc1 mol_1
O=c1[nH]c2ccccc2c(=O)o1 mol_2
O=c1cccccc1O mol_3
Nc1ncnc2c1ncn2[C@H]1CC[C@@H](CO)O1 mol_4


In [6]:
!mmpdb fragment tox21_valid.smi -o tox21_valid.fragdb
!mmpdb index tox21_valid.fragdb -o tox21_valid.mmpdb --max-radius 2
!ls -la tox21_valid.*

Fragmented record 0/1173 (0.0%)[16:03:06] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.EXPLICIT) instead
[16:03:06] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.EXPLICIT) instead
[16:03:06] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.EXPLICIT) instead
[16:03:06] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.EXPLICIT) instead
[16:03:06] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.EXPLICIT) instead
[16:03:06] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.EXPLICIT) instead
[16:03:06] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.EXPLICIT) instead
[16:03:06] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.EXPLICIT) instead
[16:03:06] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.EXPLICIT) instead
[16:03:06] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.EXPLICIT) instead
[16:03:06] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.EXPLICIT

In [7]:
import numpy as np
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from sklearn.ensemble import RandomForestClassifier

_generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
def smiles_to_ecfp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return _generator.GetFingerprintAsNumPy(mol) if mol else None

X_train, y_train, w_train = data['X_train'], data['y_train'], data['w_train']
task_cols = data['task_cols']
classifiers = {}
for i, task in enumerate(task_cols):
    train_mask = w_train[:, i] == 1
    clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
    clf.fit(X_train[train_mask], y_train[train_mask, i])
    classifiers[task] = clf

def predict_tox21_avg(s):
    m = Chem.MolFromSmiles(s)
    if not m: return None
    fp = smiles_to_ecfp(s).reshape(1,-1)
    return np.mean([classifiers[t].predict_proba(fp)[0][1] for t in task_cols])

print("Tox21 baseline 완료")

Tox21 baseline 완료


In [8]:
with open("tox21_valid_properties.csv", "w") as f:
    f.write("ID\tTox21_avg\n")
    for i, smi in enumerate(data['smiles_valid']):
        pred = predict_tox21_avg(smi)
        if pred is not None:
            f.write(f"mol_{i}\t{pred:.4f}\n")

print("완료")
!head -5 tox21_valid_properties.csv
!cat -A tox21_valid_properties.csv | head -2

완료
ID	Tox21_avg
mol_0	0.0583
mol_1	0.0572
mol_2	0.0725
mol_3	0.1125
ID^ITox21_avg$
mol_0^I0.0583$


In [9]:
!mmpdb loadprops -p tox21_valid_properties.csv tox21_valid.mmpdb

Using dataset: MMPs from 'tox21_valid.fragdb'
Reading properties from 'tox21_valid_properties.csv'
Read 1 properties for 1173 compounds from 'tox21_valid_properties.csv'
517 compounds from 'tox21_valid_properties.csv' are not in the dataset at 'tox21_valid.mmpdb'
Imported 656 'Tox21_avg' records (656 new, 0 updated).
Number of rule statistics added: 196981 updated: 0 deleted: 0
Loaded all properties and re-computed all rule statistics.


In [10]:
import sqlite3

conn = sqlite3.connect("tox21_valid.mmpdb")
cur = conn.cursor()

query = """
SELECT rs.smiles AS from_smiles, rs2.smiles AS to_smiles,
       res.count, res.avg, res.std, res.paired_t, res.p_value
FROM rule r
JOIN rule_smiles rs ON r.from_smiles_id = rs.id
JOIN rule_smiles rs2 ON r.to_smiles_id = rs2.id
JOIN rule_environment re ON re.rule_id = r.id
JOIN rule_environment_statistics res ON res.rule_environment_id = re.id
WHERE rs.smiles LIKE '%C(=O)O%'
LIMIT 20;
"""

for row in cur.execute(query):
    print(row)

conn.close()

('[*:1]C=CC(=O)OCCOCC', '[*:1]COC(C)=O', 1, -0.0542, None, None, None)
('[*:1]C=CC(=O)OCCOCC', '[*:1]COC(C)=O', 1, -0.0542, None, None, None)
('[*:1]C=CC(=O)OCCOCC', '[*:1]COC(C)=O', 1, -0.0542, None, None, None)
('[*:1]CCC(=O)C(=O)O', '[*:1]CO', 1, 0.0058000000000000005, None, None, None)
('[*:1]CCC(=O)C(=O)O', '[*:1]CO', 1, 0.0058000000000000005, None, None, None)
('[*:1]CCC(=O)C(=O)O', '[*:1]CO', 1, 0.0058000000000000005, None, None, None)
('[*:1]C(=C)CC(=O)O', '[*:1]CO', 1, -0.0225, None, None, None)
('[*:1]C(=C)CC(=O)O', '[*:1]CO', 1, -0.0225, None, None, None)
('[*:1]C(=C)CC(=O)O', '[*:1]CO', 1, -0.0225, None, None, None)
('[*:1]CC(=C)C(=O)O', '[*:1]CO', 1, -0.0225, None, None, None)
('[*:1]CC(=C)C(=O)O', '[*:1]CO', 1, -0.0225, None, None, None)
('[*:1]CC(=C)C(=O)O', '[*:1]CO', 1, -0.0225, None, None, None)
('[*:1]CCCCCCCC(=O)O', '[*:1]CO', 1, -0.027499999999999997, None, None, None)
('[*:1]CCCCCCCC(=O)O', '[*:1]CO', 1, -0.027499999999999997, None, None, None)
('[*:1]CCCCCCCC(=O)

In [11]:
all_smiles_trainvalid = list(data['smiles_train']) + list(data['smiles_valid'])
print(f"전체 분자 수(train+valid): {len(all_smiles_trainvalid)}개")

with open("tox21_trainvalid.smi", "w") as f:
    for i, smi in enumerate(all_smiles_trainvalid):
        f.write(f"{smi} mol_{i}\n")

print("저장 완료")

전체 분자 수(train+valid): 6649개
저장 완료


In [12]:
!mmpdb fragment tox21_trainvalid.smi -o tox21_trainvalid.fragdb
!mmpdb index tox21_trainvalid.fragdb -o tox21_trainvalid.mmpdb --max-radius 2
!ls -la tox21_trainvalid.*

Preparing record 1262[16:07:38] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.EXPLICIT) instead
[16:07:38] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.EXPLICIT) instead
[16:07:39] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.EXPLICIT) instead
[16:07:39] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.EXPLICIT) instead
[16:07:39] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.EXPLICIT) instead
[16:07:39] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.EXPLICIT) instead
[16:07:39] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.EXPLICIT) instead
[16:07:39] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.EXPLICIT) instead
[16:07:39] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.EXPLICIT) instead
Preparing record 1337[16:07:39] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.EXPLICIT) instead
[16:07:39] DEPRECATION WARNING: please use GetValence(Chem.ValenceTy

In [13]:
!mmpdb index tox21_trainvalid.fragdb -o tox21_trainvalid.mmpdb

Constant fragment matches 0/9702 (0.0%)[16:16:09] 

****
Pre-condition Violation
neither end atom traversed
Violation occurred on line 364 in file /project/build/temp.linux-x86_64-cpython-312/rdkit/Code/GraphMol/Canon.cpp
Failed Expression: atomVisitOrders[dblBond->getBeginAtomIdx()] > 0 || atomVisitOrders[dblBond->getEndAtomIdx()] > 0
----------
Stacktrace:
----------
****

Traceback (most recent call last):
  File "/usr/local/bin/mmpdb", line 8, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 1569, in __call__
    return self.main(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 1490, in main
    rv = self.invoke(ctx)
         ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 1970, in invoke
    return _process_result(sub_ctx.command.invoke(sub_ctx))
                           ^^^^^^^^^^^^^^^^^^^^^^^^^

In [14]:
!mmpdb index tox21_trainvalid.fragdb -o tox21_trainvalid.mmpdb --max-radius 1

In [15]:
from rdkit import Chem

pattern_cumulated = Chem.MolFromSmarts("*=*=*")  # 누적이중결합 일반 패턴

all_smiles_trainvalid = list(data['smiles_train']) + list(data['smiles_valid'])
filtered_smiles = []
excluded_count = 0

for i, smi in enumerate(all_smiles_trainvalid):
    mol = Chem.MolFromSmiles(smi)
    if mol is not None and mol.HasSubstructMatch(pattern_cumulated):
        excluded_count += 1
        continue
    filtered_smiles.append((i, smi))

print(f"제외된 누적이중결합 분자: {excluded_count}개")
print(f"남은 분자: {len(filtered_smiles)}개")

with open("tox21_trainvalid_filtered.smi", "w") as f:
    for i, smi in filtered_smiles:
        f.write(f"{smi} mol_{i}\n")

print("저장 완료")

[16:17:47] WARNING: not removing hydrogen atom without neighbors


제외된 누적이중결합 분자: 513개
남은 분자: 6136개
저장 완료


In [16]:
!mmpdb fragment tox21_trainvalid_filtered.smi -o tox21_trainvalid_filtered.fragdb
!mmpdb index tox21_trainvalid_filtered.fragdb -o tox21_trainvalid_filtered.mmpdb --max-radius 2

Preparing record 1274[16:17:56] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.EXPLICIT) instead
[16:17:56] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.EXPLICIT) instead
[16:17:56] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.EXPLICIT) instead
[16:17:56] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.EXPLICIT) instead
[16:17:56] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.EXPLICIT) instead
[16:17:56] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.EXPLICIT) instead
[16:17:56] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.EXPLICIT) instead
[16:17:56] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.EXPLICIT) instead
[16:17:56] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.EXPLICIT) instead
[16:17:56] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.EXPLICIT) instead
[16:17:56] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.EXPLICIT) instead


In [17]:
import sqlite3

conn = sqlite3.connect("tox21_valid.mmpdb")
cur = conn.cursor()

target_smarts_check = {
    "acid_halide": "%C(=O)F%",
    "Sulfonic_acid_2": "%S(=O)(=O)%",
    "thiol_2": "%S%",  # 너무 광범위해서 결과 많을 수 있음, 참고용
    "het-C-het_not_in_ring": "%OC(O)%",
    "diketo_group": "%C(=O)C(=O)%",
}

for rule, pattern in target_smarts_check.items():
    query = """
    SELECT rs.smiles AS from_smiles, rs2.smiles AS to_smiles,
           res.count, res.avg, res.std, res.p_value
    FROM rule r
    JOIN rule_smiles rs ON r.from_smiles_id = rs.id
    JOIN rule_smiles rs2 ON r.to_smiles_id = rs2.id
    JOIN rule_environment re ON re.rule_id = r.id
    JOIN rule_environment_statistics res ON res.rule_environment_id = re.id
    WHERE rs.smiles LIKE ? AND res.count >= 3
    ORDER BY res.count DESC
    LIMIT 5;
    """
    print(f"=== {rule} ===")
    rows = list(cur.execute(query, (pattern,)))
    if not rows:
        print("  표본 3개 이상인 통계 없음")
    for row in rows:
        print(f"  {row}")
    print()

conn.close()

=== acid_halide ===
  표본 3개 이상인 통계 없음

=== Sulfonic_acid_2 ===
  ('[*:1]S(=O)(=O)[O-]', '[*:1][H]', 3, -0.012499999999999999, 0.0050388490749376505, 0.05012747700357032)

=== thiol_2 ===
  ('[*:1]S(=O)(=O)[O-]', '[*:1][H]', 3, -0.012499999999999999, 0.0050388490749376505, 0.05012747700357032)

=== het-C-het_not_in_ring ===
  표본 3개 이상인 통계 없음

=== diketo_group ===
  표본 3개 이상인 통계 없음



In [18]:
from rdkit.Chem.MolStandardize import rdMolStandardize

test_cases_standardize = [
    "CC(C)OC(=S)[S-]",  # 디티오카바메이트
    "S=C(SSC(=S)N1CCCCC1)N1CCCCC1",  # 디설파이드
    "CCCCCCCCOS(=O)(=O)[O-]",  # 설페이트 에스터
]

normalizer = rdMolStandardize.Normalizer()
uncharger = rdMolStandardize.Uncharger()

for smi in test_cases_standardize:
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        print(f"{smi}: 파싱 실패")
        continue

    mol_normalized = normalizer.normalize(mol)
    smi_normalized = Chem.MolToSmiles(mol_normalized)

    mol_uncharged = uncharger.uncharge(mol_normalized)
    smi_final = Chem.MolToSmiles(mol_uncharged)

    print(f"원본:    {smi}")
    print(f"정규화:  {smi_normalized}")
    print(f"탈전하:  {smi_final}")
    print()

원본:    CC(C)OC(=S)[S-]
정규화:  CC(C)OC(=S)[S-]
탈전하:  CC(C)OC(=S)S

원본:    S=C(SSC(=S)N1CCCCC1)N1CCCCC1
정규화:  S=C(SSC(=S)N1CCCCC1)N1CCCCC1
탈전하:  S=C(SSC(=S)N1CCCCC1)N1CCCCC1

원본:    CCCCCCCCOS(=O)(=O)[O-]
정규화:  CCCCCCCCOS(=O)(=O)[O-]
탈전하:  CCCCCCCCOS(=O)(=O)O



[16:25:38] Initializing Normalizer
[16:25:38] Running Normalizer
[16:25:38] Running Uncharger
[16:25:38] Removed negative charge.
[16:25:38] Running Normalizer
[16:25:38] Running Uncharger
[16:25:38] Running Normalizer
[16:25:38] Running Uncharger
[16:25:38] Removed negative charge.


In [19]:
from rdkit.Chem.Scaffolds import MurckoScaffold
from collections import Counter

def get_scaffold(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    try:
        scaffold_mol = MurckoScaffold.GetScaffoldForMol(mol)
        return Chem.MolToSmiles(scaffold_mol)
    except Exception:
        return None

# "beta-keto/anhydride" 문제가 있는 분자군 vs 없는 분자군의 scaffold 비교
target_rule_scaffold = "quinone_A(370)"  # 오늘 안트라퀴논 등 확장이 어려웠던 규칙으로 시도

with_problem = []
without_problem = []

for s in data['smiles_valid']:
    problems = detect_toxicophores(s)
    has_target = any(p['rule_name'] == target_rule_scaffold for p in problems)
    scaffold = get_scaffold(s)
    if scaffold is None:
        continue
    if has_target:
        with_problem.append(scaffold)
    else:
        without_problem.append(scaffold)

print(f"{target_rule_scaffold} 있는 분자: {len(with_problem)}개")
print(f"{target_rule_scaffold} 없는 분자: {len(without_problem)}개")

with_counts = Counter(with_problem)
without_counts = Counter(without_problem)

print(f"\n{target_rule_scaffold} 그룹의 상위 scaffold:")
for scaf, cnt in with_counts.most_common(10):
    print(f"  {scaf[:50]}: {cnt}개 (비교군에서 {without_counts.get(scaf, 0)}개)")

quinone_A(370) 있는 분자: 7개
quinone_A(370) 없는 분자: 1166개

quinone_A(370) 그룹의 상위 scaffold:
  O=C1c2ccccc2C(=O)c2ccccc21: 4개 (비교군에서 0개)
  O=C1c2ccccc2C(=O)c2c(Nc3ccccc3)cccc21: 1개 (비교군에서 0개)
  O=C1C=CC(=O)C=C1: 1개 (비교군에서 0개)
  N=C1C=CC(=N)C=C1: 1개 (비교군에서 0개)


In [20]:
pattern_anthraquinone = Chem.MolFromSmarts("O=C1c2ccccc2C(=O)c2ccccc21")
pattern_quinonediimine = Chem.MolFromSmarts("N=C1C=CC(=N)C=C1")

test_anthra = Chem.MolFromSmiles("O=C1c2ccccc2C(=O)c2ccccc21")
test_diimine = Chem.MolFromSmiles("N=C1C=CC(=N)C=C1")

print("안트라퀴논 매치:", test_anthra.HasSubstructMatch(pattern_anthraquinone), pattern_anthraquinone.GetNumAtoms())
print("퀴논디이민 매치:", test_diimine.HasSubstructMatch(pattern_quinonediimine), pattern_quinonediimine.GetNumAtoms())

matches_anthra = test_anthra.GetSubstructMatches(pattern_anthraquinone)
print("안트라퀴논 매치 위치:", matches_anthra)
for i, idx in enumerate(matches_anthra[0]):
    print(f"  위치{i} -> idx{idx}: {test_anthra.GetAtomWithIdx(idx).GetSymbol()}")

안트라퀴논 매치: True 16
퀴논디이민 매치: True 8
안트라퀴논 매치 위치: ((0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15),)
  위치0 -> idx0: O
  위치1 -> idx1: C
  위치2 -> idx2: C
  위치3 -> idx3: C
  위치4 -> idx4: C
  위치5 -> idx5: C
  위치6 -> idx6: C
  위치7 -> idx7: C
  위치8 -> idx8: C
  위치9 -> idx9: O
  위치10 -> idx10: C
  위치11 -> idx11: C
  위치12 -> idx12: C
  위치13 -> idx13: C
  위치14 -> idx14: C
  위치15 -> idx15: C


In [21]:
!cat src/tools/replacement_library.py


REPLACEMENT_LIBRARY = {
    "nitro_group": {
        "problem_smarts": "[N+](=O)[O-]",
        "candidates": [
            {"smiles": "N", "name": "primary amine",
             "rationale": "[참고] 메트로니다졸, 니트로푸란토인, 벤즈니다졸 등 일부 "
                          "항균제/항기생충제는 니트로기의 선택적 환원 활성화 자체가 "
                          "치료 메커니즘이므로, 이런 프로드러그 설계 맥락에서는 본 "
                          "치환이 적절하지 않을 수 있음. || 극성을 유지하면서 니트로기의 "
                          "환원성 대사 중간체 생성 경로를 제거함"},
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "약물유사 골격에서 흔히 쓰이는 안정적 대체기로, 수소결합 donor/acceptor 특성을 일부 유지"},
            {"smiles": "C#N", "name": "nitrile",
             "rationale": "대사 안정성이 개선된 사례가 문헌에 다수 보고됨, 다만 극성은 다소 감소"},
        ],
    },
    "aldehyde": {
        "problem_smarts": "[CX3H1](=O)",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "알데히드의 친전자성(단백질 부가물 형성 우려)을 제거하면서 유사한 형태 유지"},
            {"smiles": "C(O)", "name": "al

In [22]:
%%writefile src/tools/replacement_library.py

REPLACEMENT_LIBRARY = {
    "nitro_group": {
        "problem_smarts": "[N+](=O)[O-]",
        "candidates": [
            {"smiles": "N", "name": "primary amine",
             "rationale": "[참고] 메트로니다졸, 니트로푸란토인, 벤즈니다졸 등 일부 "
                          "항균제/항기생충제는 니트로기의 선택적 환원 활성화 자체가 "
                          "치료 메커니즘이므로, 이런 프로드러그 설계 맥락에서는 본 "
                          "치환이 적절하지 않을 수 있음. || 극성을 유지하면서 니트로기의 "
                          "환원성 대사 중간체 생성 경로를 제거함"},
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "약물유사 골격에서 흔히 쓰이는 안정적 대체기로, 수소결합 donor/acceptor 특성을 일부 유지"},
            {"smiles": "C#N", "name": "nitrile",
             "rationale": "대사 안정성이 개선된 사례가 문헌에 다수 보고됨, 다만 극성은 다소 감소"},
        ],
    },
    "aldehyde": {
        "problem_smarts": "[CX3H1](=O)",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "알데히드의 친전자성(단백질 부가물 형성 우려)을 제거하면서 유사한 형태 유지"},
            {"smiles": "C(O)", "name": "alcohol",
             "rationale": "가장 단순한 환원형 대체, 반응성 크게 감소"},
        ],
    },
    "Michael_acceptor_1": {
        "edit_method": "atom_edit",
        "problem_smarts": "C=CC(=O)",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "saturated (C-C single bond)",
             "rationale": "[참고] 에타크린산처럼 시스테인 잔기와의 공유결합 자체가 "
                          "작용 메커니즘인 공유결합 억제제(covalent inhibitor) "
                          "계열에는 본 경고가 그대로 적용되지 않을 수 있음. || "
                          "알파,베타-불포화 카르보닐의 C=C 이중결합을 환원하여 "
                          "단백질 친전자성 부가반응(Michael addition, covalent "
                          "binding) 위험을 제거함"},
        ],
    },
    "acid_halide": {
        "problem_smarts": "C(=O)[F,Cl,Br,I]",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "고반응성 아실할라이드를 안정적인 아마이드로 대체"},
            {"smiles": "C(=O)O", "name": "ester",
             "rationale": "아마이드보다 극성이 낮고 유연한 대체 옵션, 가수분해 속도 조절 가능 (검증 필요)"},
        ],
    },
    "alkyl_halide": {
        "problem_smarts": "[Cl,Br,I]",
        "candidates": [
            {"smiles": "O", "name": "hydroxyl (alcohol)",
             "rationale": "[참고] 메클로르에타민, 사이클로포스파미드, 카머스틴, "
                          "클로람부실 등 알킬화 항암제는 DNA 알킬화(반응성) 자체가 "
                          "세포독성 치료 메커니즘이므로, 이 계열에는 본 치환이 "
                          "적절하지 않음. || 이탈기를 제거해 알킬화 반응성을 없앰, "
                          "극성은 유사하게 유지"},
            {"smiles": "F", "name": "fluorine",
             "rationale": "할로겐을 유지하되 C-F 결합은 강해 이탈기로 작용하지 않음, 입체적 크기도 유사"},
        ],
    },
    "aniline": {
        "edit_method": "atom_edit",
        "problem_smarts": "[NH2]c1ccc([#6,#7,#8,#16])cc1",
        "target_idx_in_pattern": 0,
        "ring_atom_indices_in_pattern": [1, 2, 3, 4, 6, 7],
        "anchor_indices_in_pattern": (0, 5),
        "candidates": [
            {"edit_type": "add_substituent", "param": "C(=O)C",
             "target_idx_in_pattern": 0,
             "name": "acetamide (acylated amine)",
             "rationale": "[참고] 설파계 항생제(설파닐아마이드, 설파메톡사졸 등)와 "
                          "프로카인아마이드처럼 아닐린 골격이 반응성 대사가 아닌 "
                          "안정적 형태로 널리 처방되어 온 사례가 다수 있음. 이 경우 "
                          "특이체질 반응은 드물고 예측이 어려워, 본 경고를 절대적 "
                          "배제 기준이 아닌 참고 신호로 해석해야 함. || 1차 방향족 "
                          "아민을 아마이드로 아실화하여 N-hydroxylation 경로 자체를 차단"},
            {"edit_type": "replace_ring", "param": "[*:1]C12CC(C1)(C2)[*:2]",
             "ring_atom_indices_in_pattern": [1, 2, 3, 4, 6, 7],
             "anchor_indices_in_pattern": (0, 5),
             "name": "BCP (bicyclo[1.1.1]pentane)",
             "rationale": "para-이치환 아닐린의 방향족 벤젠 고리를 포화 bicyclic "
                          "탄소골격(BCP)으로 교체함. 방향족성 제거로 aniline reactive "
                          "metabolite(RM) 형성 및 CYP-inhibition을 감소시켜, 퀴논이민 "
                          "생성 경로를 차단하고 특이체질 약물 부작용(IADR) 위험을 낮춤 "
                          "(문헌 근거, 학생 제공). 벤젠과의 공간적 유사성, Fsp3 증가, "
                          "실제 성공 사례가 많아 채택. 아마이드화(단순 아민 치환)보다 "
                          "변화 폭이 크지만, 물성 개선 효과도 더 큼"},
        ],
    },
    "Sulfonic_acid_2": {
        "problem_smarts": "[#6]S(=O)(=O)[OX2H1,OX1-]",
        "candidates": [
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "[참고] 암페타민 설페이트, 사퀴나비르 메실레이트처럼 "
                          "일부 승인약물에서 설폰산/설폰산 유사기는 활성 골격이 "
                          "아니라 염(salt) 형성을 위한 카운터이온으로만 존재함. "
                          "이 경우 본 규칙이 다루는 '독성 유발 골격'과 무관하므로, "
                          "치환 대상 여부를 판단하기 전에 이 산이 활성 골격의 "
                          "일부인지 염 형성용인지 구분이 필요함. || 생리적 pH에서 "
                          "이온화 정도(전하)를 크게 낮춰 세포막 투과성을 개선함. "
                          "설폰산은 대부분 음이온 상태로 존재해 경구 흡수가 저해되는 "
                          "경우가 많으나, 설폰아마이드는 유사한 골격을 유지하면서도 "
                          "중성에 가까워 약물유사성이 개선됨"},
            {"smiles": "C(=O)O", "name": "carboxylic acid",
             "rationale": "설폰산보다 산성도가 약하고 부피가 작은 산성 bioisostere "
                          "(검증 필요)"},
        ],
    },
    "imine_1_oxime": {
        "edit_method": "atom_edit",
        "problem_smarts": "C=N[OX2H1]",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "amine (reduced)",
             "rationale": "옥심의 C=N 결합을 환원하여, 가수분해 시 원래의 반응성 "
                          "카르보닐(알데히드/케톤)로 되돌아갈 수 있는 대사 불안정 "
                          "경로를 제거함"},
        ],
    },
    "imine_1_general": {
        "edit_method": "atom_edit",
        "problem_smarts": "[CX3;!$(C(N)(N)=N)]=N",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "amine (reduced)",
             "rationale": "일반 이민(C=N-R)을 환원하여 가수분해 시 반응성 카르보닐로 "
                          "되돌아갈 수 있는 대사 불안정 경로를 제거함. 옥심 특유의 "
                          "메커니즘보다는 근거가 다소 약하며, 하위 구조별 개별 검증 필요. "
                          "구아니딘(N-C(=N)-N, 공명구조로 일반 이민과 반응성이 다름)은 "
                          "이 SMARTS에서 명시적으로 제외함"},
        ],
    },
    "catechol": {
        "edit_method": "atom_edit",
        "problem_smarts": "[OX2H;$(Oc1ccccc1O)]",
        "target_idx_in_pattern": 0,
        "candidates": [
            {"edit_type": "add_substituent", "param": "C", "name": "methoxy",
             "rationale": "[참고] 도파민, 에피네프린, 이소프로테레놀 등 카테콜아민류 "
                          "약물은 카테콜 구조 자체가 아드레날린/도파민 수용체 결합에 "
                          "필수적인 약효 골격이므로, 이 경우 본 치환은 독성 감소가 "
                          "아니라 약효 상실로 이어짐. || 인체의 COMT(catechol-O-"
                          "methyltransferase) 효소가 카테콜을 메톡시페놀로 메틸화하여 "
                          "해독하는 생리적 경로와 동일한 원리. 오르토-퀴논으로의 산화 "
                          "경로를 차단하여 세포독성/유전독성 우려를 낮춤 (학생 확인 "
                          "예정: ScienceDirect catechol overview, PMC6643002 등 참고)"},
        ],
    },
    "Thiocarbonyl_group": {
        "edit_method": "atom_edit",
        "problem_smarts": "[#6]=[#16]",
        "target_idx_in_pattern": 1,
        "candidates": [
            {"edit_type": "replace_element", "param": 8, "name": "carbonyl (O replacing S)",
             "rationale": "[참고] 티오펜탈·티아밀랄(치오바르비투레이트, C=S가 지용성 "
                          "증가로 빠른 마취효과에 기여)과 티오구아닌(퓨린 유사 항대사물, "
                          "황이 작용기전에 필수)처럼 황 원자가 약효/효력에 직접 "
                          "기여하는 경우가 있어, 이 계열에는 본 치환이 부적절할 수 "
                          "있음. || 황을 산소로 대체(티오카르보닐->카르보닐)하는 것은 "
                          "흔한 bioisostere 전략으로, 갑상선 기능 저해 등 황 함유 "
                          "작용기 특유의 대사/독성 우려를 낮춤 (검증 필요, "
                          "thiourea->urea 치환 논리와 동일 계열)"},
        ],
    },
    "thiol_2": {
        "problem_smarts": "[SX2H1]",
        "candidates": [
            {"smiles": "O", "name": "hydroxyl (alcohol)",
             "rationale": "티올의 금속 킬레이팅 및 산화(이황화물/술펜산 형성) 반응성을 "
                          "제거하면서, 극성·수소결합 특성을 유사하게 유지함"},
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "티올을 아마이드로 대체하여 반응성을 낮추면서 약물유사 골격에서 "
                          "흔히 쓰이는 안정적 작용기로 전환 (검증 필요)"},
        ],
    },
    "thiol_1_dithiocarbamate": {
        "edit_method": "atom_edit",
        "problem_smarts": "C(=S)[SX1-]",
        "candidates": [
            {"edit_type": "replace_multi",
             "param": [
                 {"idx_in_pattern": 1, "new_element": 8, "new_charge": 0},
                 {"idx_in_pattern": 2, "new_element": 7, "new_charge": 0},
             ],
             "name": "carbamate (O,N replacing S,S)",
             "rationale": "디티오카바메이트(R-O-C(=S)-S-)를 카바메이트(R-O-C(=O)-N)로 "
                          "전환. 두 황 원자를 각각 산소·질소로 교체하여 금속 킬레이팅 "
                          "능력과 효소 억제 활성(디티오카바메이트류 특유의 살충제성 "
                          "독성 기전)을 제거함 (검증 필요)"},
        ],
    },
    "thiol_1_thiocarboxylate": {
        "edit_method": "atom_edit",
        "problem_smarts": "[SX1-]C(=O)",
        "target_idx_in_pattern": 0,
        "candidates": [
            {"edit_type": "replace_element", "param": 8, "name": "carboxylate (O replacing S)",
             "rationale": "티오카르복실산 음이온(R-C(=O)-S-)의 황을 산소로 대체하여 "
                          "카르복실산염(R-C(=O)-O-)으로 전환. 황 원자의 금속 킬레이팅 "
                          "및 친핵성 반응성을 제거함 (검증 필요)"},
        ],
    },
    "het-C-het_not_in_ring": {
        "edit_method": "atom_edit",
        "problem_smarts": "[CX4](O)(O)",
        "candidates": [
            {"edit_type": "remove_substituent",
             "center_idx_in_pattern": 0,
             "remove_idx_in_pattern": 1,
             "upgrade_bond_to_idx_in_pattern": 2,
             "name": "ketone/ester (one alkoxy removed, C=O formed)",
             "rationale": "아세탈/케탈 또는 오르토에스터(탄소 하나에 알콕시기 2개 "
                          "이상)는 가수분해에 민감하여 반응성 카르보닐(케톤/알데히드)로 "
                          "쉽게 분해되며 대사 불안정성을 일으킴. 알콕시기 하나를 제거하고 "
                          "남은 산소를 카르보닐로 승격시켜, 가수분해로 어차피 도달할 "
                          "안정한 최종 형태로 미리 전환함 (검증 필요)"},
        ],
    },
    "hydroquinone": {
        "edit_method": "atom_edit",
        "problem_smarts": "[OX2H]c1ccc([OX2H,NX3H1,NX3H2])cc1",
        "target_idx_in_pattern": 0,
        "candidates": [
            {"edit_type": "add_substituent", "param": "C", "name": "methoxy",
             "rationale": "[참고] 아세트아미노펜은 정상 용량에서는 안전하며 과다복용 "
                          "시에만 위험한 용량 의존적 사례임. 본 시스템은 치료지수를 "
                          "고려하지 않으므로, 아트로핀·디곡신·와파린처럼 좁은 치료지수를 "
                          "가진 기존 약물 전반에 유사하게 적용되는 한계임. || 파라 "
                          "위치에 OH와 (OH 또는 NH)가 있는 구조(하이드로퀴논/파라-"
                          "아미노페놀 계열)는 산화되어 파라-퀴논 또는 파라-퀴논이민(예: "
                          "아세트아미노펜의 NAPQI)을 형성, 글루타치온 고갈과 단백질 "
                          "공유결합을 통한 간독성 위험이 있음"},
        ],
    },
    "azo_A(324)": {
        "edit_method": "atom_edit",
        "problem_smarts": "N=N",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "hydrazine (reduced)",
             "rationale": "아조기(N=N)는 체내에서 아조환원효소에 의해 환원되어 두 개의 "
                          "방향족 아민으로 분해되며, 그 중 일부(벤지딘류 등)가 발암성을 "
                          "가지는 것으로 잘 알려짐(아조 색소의 대표적 독성 메커니즘). "
                          "이중결합을 환원하여 하이드라진 형태로 전환, 완전한 아민 "
                          "분해 경로 자체를 차단함 (검증 필요: 하이드라진 자체의 "
                          "잔여 반응성은 추가 확인 필요)"},
        ],
    },
    "Three-membered_heterocycle": {
        "edit_method": "atom_edit",
        "problem_smarts": "[CX4]1[OX2][CX4]1",
        "candidates": [
            {"edit_type": "open_epoxide", "break_pair_in_pattern": (1, 2),
             "name": "vicinal diol (ring-opened)",
             "rationale": "에폭시드(3원자 고리, 옥시란)는 고리 변형(strain)으로 인해 "
                          "친핵체(DNA, 단백질)와 쉽게 반응하는 알킬화제로 작용함. "
                          "체내 에폭시드 가수분해효소(epoxide hydrolase)가 실제로 "
                          "수행하는 반응과 동일하게 고리를 열어 비시날 디올(vicinal "
                          "diol)로 전환, 반응성을 제거함"},
        ],
    },
    "diketo_group": {
        "edit_method": "atom_edit",
        "problem_smarts": "C(=O)C(=O)",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "alpha-hydroxy ketone (reduced)",
             "rationale": "비시날 알파-디케톤(1,2-diketone)은 반응성이 높은 친전자체로 "
                          "단백질과 부가물을 형성할 수 있으며, 흡입 시 호흡기 독성을 "
                          "일으키는 것으로 알려진 디아세틸(버터향 첨가제) 사례가 대표적임. "
                          "카르보닐 하나를 환원하여 알파-하이드록시케톤(아실로인)으로 "
                          "전환, 케토-환원효소에 의한 실제 해독 경로와 유사한 방향으로 "
                          "반응성을 낮춤 (검증 필요)"},
        ],
    },
    "thioester": {
        "edit_method": "atom_edit",
        "problem_smarts": "[SX2](C(=O))",
        "target_idx_in_pattern": 0,
        "candidates": [
            {"edit_type": "replace_element", "param": 8, "name": "ester (O replacing S)",
             "rationale": "티오에스터의 황을 산소로 대체하여 일반 에스터로 전환. "
                          "티오에스터는 일반 에스터보다 가수분해 반응성이 높고 아실화 "
                          "능력이 강해 단백질 등과 부반응 우려가 있음 (검증 필요)"},
        ],
    },
    "N-nitroso": {
        "edit_method": "atom_edit",
        "problem_smarts": "[NX2;+0;!$(N(=O)[O-])]=[OX1;+0]",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "N-hydroxylamine (reduced)",
             "rationale": "N-니트로소 화합물(니트로사민)은 대사 활성화(알파-수산화)를 "
                          "거쳐 강력한 알킬화 발암물질을 생성하는 것으로 잘 알려짐 "
                          "(발사르탄, 라니티딘 등 실제 의약품 불순물 리콜 사례). "
                          "N=O를 환원하여 반응성을 낮춤 (검증 필요: 완전한 해독은 "
                          "탈니트로소화가 필요하며 이는 근사적 접근)"},
        ],
    },
    "hydrazine": {
        "edit_method": "atom_edit",
        "problem_smarts": "[NX3H2][NX3H1]",
        "center_idx_in_pattern": 1,
        "candidates": [
            {"edit_type": "remove_atom",
             "remove_idx_in_pattern": 0,
             "center_idx_in_pattern": 1,
             "name": "amide/amine (terminal N removed)",
             "rationale": "하이드라진/하이드라지드(R-NH-NH2)의 말단 질소를 제거하여 "
                          "단순 아민 또는 아마이드로 되돌림. 하이드라진류는 대사 시 "
                          "반응성 디아제늄 중간체를 형성해 유전독성을 일으킬 수 있는 "
                          "것으로 알려짐. 이는 azo_A(324) 환원 시 생성되는 하이드라진 "
                          "중간체의 잔여 위험을 추가로 낮추는 후속 규칙이기도 함 "
                          "(검증 필요)"},
        ],
    },
    "sulphate": {
        "edit_method": "atom_edit",
        "problem_smarts": "[OX2][SX4](=O)(=O)[OX1,OX2H]",
        "center_idx_in_pattern": 0,
        "candidates": [
            {"edit_type": "remove_atom",
             "remove_idx_in_pattern": 1,
             "center_idx_in_pattern": 0,
             "name": "alcohol (sulfate group removed)",
             "rationale": "알킬 설페이트 에스터(R-O-SO3-)는 대사되어 반응성 있는 "
                          "설페이트 이탈기를 통한 알킬화제로 작용할 수 있음(디메틸설페이트가 "
                          "강력한 발암/독성 물질로 잘 알려진 대표 사례). 설페이트기 전체를 "
                          "제거하여 원래의 알코올로 되돌림 (검증 필요)"},
        ],
    },
    "N_oxide": {
        "edit_method": "atom_edit",
        "problem_smarts": "[n+][O-]",
        "center_idx_in_pattern": 0,
        "candidates": [
            {"edit_type": "remove_atom",
             "remove_idx_in_pattern": 1,
             "center_idx_in_pattern": 0,
             "name": "pyridine (N-oxide removed)",
             "rationale": "방향족 N-옥사이드는 산화적 대사산물이자 반응성 중간체 "
                          "생성 경로의 일부일 수 있음. 산소를 제거하여 원래의 중성 "
                          "방향족 아민(피리딘 등)으로 환원, 자연 대사에서의 환원 "
                          "경로와 유사한 방향으로 반응성을 낮춤 (검증 필요)"},
        ],
    },
    "2-halo_pyridine": {
        "edit_method": "atom_edit",
        "problem_smarts": "n:c(-[Cl,Br,I])",
        "center_idx_in_pattern": 1,
        "candidates": [
            {"edit_type": "remove_atom",
             "remove_idx_in_pattern": 2,
             "center_idx_in_pattern": 1,
             "name": "pyridine (halogen removed)",
             "rationale": "피리딘 고리 질소에 인접한 위치의 할로겐(특히 불소/염소)은 "
                          "친핵성 방향족 치환(SNAr) 반응에 취약해, 체내 친핵체(글루타치온, "
                          "단백질 시스테인 등)와 반응할 수 있음. 할로겐을 제거하고 수소로 "
                          "대체하여 이 반응성 경로를 차단함 (검증 필요)"},
        ],
    },
    "disulphide": {
        "edit_method": "atom_edit",
        "problem_smarts": "[SX2][SX2]",
        "candidates": [
            {"edit_type": "cleave_bond", "cleave_pair_in_pattern": (0, 1),
             "name": "two thiols (bond cleaved)",
             "rationale": "[참고] 이황화결합(S-S)은 시스틴/단백질의 3차구조 형성에 "
                          "필수적인 정상 생체 구조이기도 하므로, 이 결합이 약물의 "
                          "구조 안정성이나 표적 결합에 관여하는 경우 본 치환이 "
                          "부적절할 수 있음. || 디티오카바메이트류(티우람 등) 농약/"
                          "살균제에서 흔한 반응성 이황화결합을 두 개의 티올로 분리, "
                          "산화·금속킬레이팅 반응성을 낮춤 (검증 필요)"},
        ],
    },
    "quinone_A(370)": {
        "edit_method": "atom_edit",
        "problem_smarts": "O=C1C=CC(=O)C=C1",
        "target_pairs_in_pattern": [(1, 0), (4, 5)],
        "ring_atoms_in_pattern": [1, 2, 3, 4, 6, 7],
        "ring_bonds_in_pattern": [(1, 2), (2, 3), (3, 4), (4, 6), (6, 7), (7, 1)],
        "candidates": [
            {"edit_type": "reduce_multi_bond", "name": "hydroquinone (reduced, re-aromatized)",
             "rationale": "파라벤조퀴논은 산화환원 사이클(redox cycling)을 통해 활성산소종(ROS)을 "
                          "생성하고 DNA/단백질과 직접 공유결합하는 대표적 반응성 구조. 체내 "
                          "NQO1(퀴논 환원효소) 효소가 실제로 수행하는 반응과 동일하게 두 카르보닐을 "
                          "환원하고 고리를 재방향족화하여 안정적인 하이드로퀴논으로 전환. 결과물이 "
                          "다시 hydroquinone 규칙에 해당할 수 있으며, 이 경우 반복 루프가 자동으로 "
                          "메톡시페놀 등 산화에 더 안정적인 형태로 한 단계 더 개선함 (검증 필요, "
                          "안트라퀴논 등 융합고리형은 미지원)"},
        ],
    },
    "quinone_A_anthraquinone": {
        "edit_method": "atom_edit",
        "problem_smarts": "O=C1c2ccccc2C(=O)c2ccccc21",
        "target_pairs_in_pattern": [(1, 0), (8, 9)],
        "ring_atoms_in_pattern": [1, 2, 3, 4, 5, 6, 7, 8],
        "ring_bonds_in_pattern": [(1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7), (7, 8), (8, 1)],
        "candidates": [
            {"edit_type": "reduce_multi_bond", "name": "anthrahydroquinone (reduced, re-aromatized)",
             "rationale": "안트라퀴논은 벤조퀴논과 동일한 산화환원 사이클링(redox cycling) 메커니즘을 "
                          "가지되, 두 벤젠 고리에 의해 안정화되어 항암제(독소루비신 등) 및 염료에서도 "
                          "흔히 쓰이는 골격임. 두 카르보닐을 동시에 환원하고 중앙 고리를 재방향족화하여 "
                          "안트라하이드로퀴논으로 전환, 산화환원 사이클링 능력을 제거함. 결과물이 "
                          "hydroquinone 규칙에 해당할 수 있어 반복 루프가 자동으로 추가 개선 가능 "
                          "(Murcko scaffold 분석으로 발견, 검증 필요)"},
        ],
    },
    "quinone_diimine": {
        "edit_method": "atom_edit",
        "problem_smarts": "N=C1C=CC(=N)C=C1",
        "target_pairs_in_pattern": [(1, 0), (4, 5)],
        "ring_atoms_in_pattern": [1, 2, 3, 4, 6, 7],
        "ring_bonds_in_pattern": [(1, 2), (2, 3), (3, 4), (4, 6), (6, 7), (7, 1)],
        "candidates": [
            {"edit_type": "reduce_multi_bond", "name": "phenylenediamine (reduced, re-aromatized)",
             "rationale": "퀴논디이민(quinone diimine)은 벤조퀴논의 산소가 이민으로 치환된 유사체로, "
                          "동일한 산화환원 사이클링 메커니즘을 가지며 헤어염료 성분(파라페닐렌디아민 "
                          "산화형) 등에서 피부 알레르기 및 접촉성 피부염을 유발하는 것으로 알려짐. 두 "
                          "이민을 동시에 환원하고 고리를 재방향족화하여 페닐렌디아민(원래의 안정한 "
                          "환원형)으로 전환 (Murcko scaffold 분석으로 발견, 검증 필요)"},
        ],
    },
    "isocyanate": {
        "edit_method": "atom_edit",
        "problem_smarts": "[NX2]=[CX2]=[OX1]",
        "center_idx_in_pattern": 0,
        "candidates": [
            {"edit_type": "remove_atom",
             "remove_idx_in_pattern": 1,
             "center_idx_in_pattern": 0,
             "name": "amine (NCO hydrolyzed)",
             "rationale": "이소시아네이트(R-N=C=O)는 매우 반응성이 높은 친전자체로, "
                          "단백질/아미노기와 쉽게 부가반응을 일으켜 직업성 천식·과민증을 "
                          "유발하는 것으로 잘 알려짐(TDI, MDI 등 산업용 이소시아네이트 "
                          "사례). 체내/환경에서 실제로 일어나는 가수분해 경로(R-NCO + H2O "
                          "-> R-NH2 + CO2)와 동일하게 카르보닐 탄소와 산소를 제거하고 "
                          "질소만 남겨 아민으로 전환 (검증 필요)"},
        ],
    },
    "triple_bond": {
        "problem_smarts": "C#C",
        "edit_method": "atom_edit",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "alkene (partially reduced)",
             "rationale": "말단 알카인(삼중결합)은 CYP450 효소에 의해 기계기반 억제"
                          "(mechanism-based inhibition) 경로로 대사되며, 반응성 케텐/"
                          "에폭사이드 중간체를 형성해 효소를 비가역적으로 불활성화할 "
                          "수 있음(에티닐에스트라디올 등에서 알려진 메커니즘). 삼중결합을 "
                          "이중결합으로 환원하여 반응성을 낮춤 (검증 필요, 완전 포화가 "
                          "아닌 부분 환원)"},
        ],
    },
    "stilbene": {
        "problem_smarts": "c-[CX3]=[CX3]-c",
        "edit_method": "atom_edit",
        "target_idx_pair_in_pattern": (1, 2),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "diarylethane (reduced)",
             "rationale": "스틸벤 구조(두 방향족 고리를 잇는 C=C)는 디에틸스틸베스트롤"
                          "(DES)처럼 내분비교란 및 대사 산화를 통한 반응성 중간체 형성이 "
                          "알려진 골격. 이중결합을 환원하여 평면성을 낮추고 대사 반응성을 "
                          "완화함 (검증 필요, 에스트로겐 수용체 결합에 필요한 형태 자체를 "
                          "훼손할 수 있어 신중한 해석 필요)"},
        ],
    },
    "beta-keto/anhydride": {
        "edit_method": "atom_edit",
        "problem_smarts": "C(=O)OC(=O)",
        "center_idx_in_pattern": 2,
        "candidates": [
            {"edit_type": "remove_atom",
             "remove_idx_in_pattern": 3,
             "center_idx_in_pattern": 2,
             "name": "carboxylic acid (anhydride hydrolyzed)",
             "rationale": "산 무수물(R-C(=O)-O-C(=O)-R')은 강한 아실화제로 단백질 아미노산 "
                          "잔기와 쉽게 반응하며, 수용액 환경에서 자발적으로 가수분해되어 "
                          "두 개의 카르복실산으로 분해되는 것이 자연스러운 무독화 경로임. "
                          "한쪽 아실기를 제거하여 이 가수분해 최종형(카르복실산)으로 직접 "
                          "전환 (검증 필요). ※ 대안 후보(무수물->아마이드/이미드 bioisostere) "
                          "는 문헌 확인 후 추가 예정"},
        ],
    },
}

def get_replacement_candidates(rule_name: str) -> dict | None:
    """rule_name에 해당하는 치환 정보(SMARTS + 후보 리스트)를 반환. 없으면 None."""
    return REPLACEMENT_LIBRARY.get(rule_name)


Overwriting src/tools/replacement_library.py


In [23]:
importlib.reload(src.tools.replacement_library)
importlib.reload(src.tools.atom_editor)
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import propose_fix

print("안트라퀴논:", propose_fix("O=C1c2ccccc2C(=O)c2ccccc21", "quinone_A_anthraquinone", candidate_idx=0))
print("퀴논디이민:", propose_fix("N=C1C=CC(=N)C=C1", "quinone_diimine", candidate_idx=0))

print("\n=== 회귀 테스트 ===")
print(propose_fix("O=C1C=CC(=O)C=C1", "quinone_A(370)", candidate_idx=0))

안트라퀴논: None
퀴논디이민: {'new_smiles': 'Nc1ccc(N)cc1', 'candidate_used': 'phenylenediamine (reduced, re-aromatized)', 'rationale': '퀴논디이민(quinone diimine)은 벤조퀴논의 산소가 이민으로 치환된 유사체로, 동일한 산화환원 사이클링 메커니즘을 가지며 헤어염료 성분(파라페닐렌디아민 산화형) 등에서 피부 알레르기 및 접촉성 피부염을 유발하는 것으로 알려짐. 두 이민을 동시에 환원하고 고리를 재방향족화하여 페닐렌디아민(원래의 안정한 환원형)으로 전환 (Murcko scaffold 분석으로 발견, 검증 필요)', 'is_valid': True}

=== 회귀 테스트 ===
{'new_smiles': 'Oc1ccc(O)cc1', 'candidate_used': 'hydroquinone (reduced, re-aromatized)', 'rationale': '파라벤조퀴논은 산화환원 사이클(redox cycling)을 통해 활성산소종(ROS)을 생성하고 DNA/단백질과 직접 공유결합하는 대표적 반응성 구조. 체내 NQO1(퀴논 환원효소) 효소가 실제로 수행하는 반응과 동일하게 두 카르보닐을 환원하고 고리를 재방향족화하여 안정적인 하이드로퀴논으로 전환. 결과물이 다시 hydroquinone 규칙에 해당할 수 있으며, 이 경우 반복 루프가 자동으로 메톡시페놀 등 산화에 더 안정적인 형태로 한 단계 더 개선함 (검증 필요, 안트라퀴논 등 융합고리형은 미지원)', 'is_valid': True}


In [24]:
info_anthra = get_replacement_candidates("quinone_A_anthraquinone")
print("info:", info_anthra)

pattern_check = Chem.MolFromSmarts(info_anthra['problem_smarts'])
mol_check = Chem.MolFromSmiles("O=C1c2ccccc2C(=O)c2ccccc21")
print("매치:", mol_check.HasSubstructMatch(pattern_check))

info: {'edit_method': 'atom_edit', 'problem_smarts': 'O=C1c2ccccc2C(=O)c2ccccc21', 'target_pairs_in_pattern': [(1, 0), (8, 9)], 'ring_atoms_in_pattern': [1, 2, 3, 4, 5, 6, 7, 8], 'ring_bonds_in_pattern': [(1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7), (7, 8), (8, 1)], 'candidates': [{'edit_type': 'reduce_multi_bond', 'name': 'anthrahydroquinone (reduced, re-aromatized)', 'rationale': '안트라퀴논은 벤조퀴논과 동일한 산화환원 사이클링(redox cycling) 메커니즘을 가지되, 두 벤젠 고리에 의해 안정화되어 항암제(독소루비신 등) 및 염료에서도 흔히 쓰이는 골격임. 두 카르보닐을 동시에 환원하고 중앙 고리를 재방향족화하여 안트라하이드로퀴논으로 전환, 산화환원 사이클링 능력을 제거함. 결과물이 hydroquinone 규칙에 해당할 수 있어 반복 루프가 자동으로 추가 개선 가능 (Murcko scaffold 분석으로 발견, 검증 필요)'}]}
매치: True


In [25]:
mol_debug = Chem.MolFromSmiles("O=C1c2ccccc2C(=O)c2ccccc21")
match_debug = mol_debug.GetSubstructMatches(pattern_check)[0]
print("match:", match_debug)

rwmol_debug = Chem.RWMol(mol_debug)

pairs = [(1, 0), (8, 9)]
for pair in pairs:
    idx_c = match_debug[pair[0]]
    idx_o = match_debug[pair[1]]
    bond = rwmol_debug.GetBondBetweenAtoms(idx_c, idx_o)
    print(f"pair {pair} -> atoms ({idx_c},{idx_o}), bond exists: {bond is not None}")

match: (0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15)
pair (1, 0) -> atoms (1,0), bond exists: True
pair (8, 9) -> atoms (8,9), bond exists: True


In [26]:
# 카르보닐 환원 계속 진행
for pair in pairs:
    idx_c = match_debug[pair[0]]
    idx_o = match_debug[pair[1]]
    bond = rwmol_debug.GetBondBetweenAtoms(idx_c, idx_o)
    bond.SetBondType(Chem.BondType.SINGLE)
    rwmol_debug.GetAtomWithIdx(idx_o).SetNoImplicit(False)
    rwmol_debug.GetAtomWithIdx(idx_c).SetNumExplicitHs(0)
    rwmol_debug.GetAtomWithIdx(idx_c).SetNoImplicit(False)

ring_atoms_pattern = [1, 2, 3, 4, 5, 6, 7, 8]
ring_indices = [match_debug[i] for i in ring_atoms_pattern]
print("ring_indices:", ring_indices)

for a in ring_indices:
    rwmol_debug.GetAtomWithIdx(a).SetIsAromatic(True)

ring_bonds_pattern = [(1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7), (7, 8), (8, 1)]
for b1, b2 in ring_bonds_pattern:
    bidx1, bidx2 = match_debug[b1], match_debug[b2]
    rbond = rwmol_debug.GetBondBetweenAtoms(bidx1, bidx2)
    print(f"ring bond ({b1},{b2}) -> atoms ({bidx1},{bidx2}), exists: {rbond is not None}")
    if rbond is not None:
        rbond.SetBondType(Chem.BondType.AROMATIC)
        rbond.SetIsAromatic(True)

try:
    new_mol_debug = rwmol_debug.GetMol()
    Chem.SanitizeMol(new_mol_debug)
    print("성공:", Chem.MolToSmiles(new_mol_debug))
except Exception as e:
    print("실패:", repr(e))

ring_indices: [1, 2, 3, 4, 5, 6, 7, 8]
ring bond (1,2) -> atoms (1,2), exists: True
ring bond (2,3) -> atoms (2,3), exists: True
ring bond (3,4) -> atoms (3,4), exists: True
ring bond (4,5) -> atoms (4,5), exists: True
ring bond (5,6) -> atoms (5,6), exists: True
ring bond (6,7) -> atoms (6,7), exists: True
ring bond (7,8) -> atoms (7,8), exists: True
ring bond (8,1) -> atoms (8,1), exists: False
성공: Oc1c2ccccc2c(O)c2ccccc12


In [27]:
!cat src/tools/atom_editor.py

from rdkit import Chem


def apply_atom_edit_from_rule(smiles: str, rule_name: str, candidate_idx: int = 0):
    """replacement_library의 atom_edit 규칙을 이용해 원자/결합/고리 직접 편집을 수행."""
    from src.tools.replacement_library import get_replacement_candidates
    info = get_replacement_candidates(rule_name)
    if info is None or info.get("edit_method") != "atom_edit":
        return None
    if candidate_idx >= len(info["candidates"]):
        return None

    candidate = info["candidates"][candidate_idx]
    smarts = info["problem_smarts"]

    mol = Chem.MolFromSmiles(smiles)
    pattern = Chem.MolFromSmarts(smarts)
    if mol is None or pattern is None:
        return None

    matches = mol.GetSubstructMatches(pattern)
    if not matches:
        return None
    match = matches[0]

    rwmol = Chem.RWMol(mol)
    edit_type = candidate["edit_type"]

    if edit_type == "replace_element":
        target_idx = match[candidate.get("target_idx_in_pattern", info.get("target_idx_in_pattern"))]
  

In [28]:
%%writefile src/tools/atom_editor.py

from rdkit import Chem


def apply_atom_edit_from_rule(smiles: str, rule_name: str, candidate_idx: int = 0):
    """replacement_library의 atom_edit 규칙을 이용해 원자/결합/고리 직접 편집을 수행."""
    from src.tools.replacement_library import get_replacement_candidates
    info = get_replacement_candidates(rule_name)
    if info is None or info.get("edit_method") != "atom_edit":
        return None
    if candidate_idx >= len(info["candidates"]):
        return None

    candidate = info["candidates"][candidate_idx]
    smarts = info["problem_smarts"]

    mol = Chem.MolFromSmiles(smiles)
    pattern = Chem.MolFromSmarts(smarts)
    if mol is None or pattern is None:
        return None

    matches = mol.GetSubstructMatches(pattern)
    if not matches:
        return None
    match = matches[0]

    rwmol = Chem.RWMol(mol)
    edit_type = candidate["edit_type"]

    if edit_type == "replace_element":
        target_idx = match[candidate.get("target_idx_in_pattern", info.get("target_idx_in_pattern"))]
        atom = rwmol.GetAtomWithIdx(target_idx)
        atom.SetAtomicNum(candidate["param"])

    elif edit_type == "add_substituent":
        target_idx = match[candidate.get("target_idx_in_pattern", info.get("target_idx_in_pattern"))]
        frag = Chem.MolFromSmiles(candidate["param"])
        if frag is None:
            return None
        combined = Chem.CombineMols(rwmol.GetMol(), frag)
        rwmol = Chem.RWMol(combined)
        offset = mol.GetNumAtoms()
        rwmol.AddBond(target_idx, offset, Chem.BondType.SINGLE)
        atom = rwmol.GetAtomWithIdx(target_idx)
        if atom.GetNumExplicitHs() > 0:
            atom.SetNumExplicitHs(atom.GetNumExplicitHs() - 1)
        else:
            atom.SetNoImplicit(False)

    elif edit_type == "reduce_bond":
        pair = candidate.get("target_idx_pair_in_pattern", info.get("target_idx_pair_in_pattern"))
        idx1 = match[pair[0]]
        idx2 = match[pair[1]]
        bond = rwmol.GetBondBetweenAtoms(idx1, idx2)
        if bond is None:
            return None
        bond.SetBondType(Chem.BondType.SINGLE)
        for idx in (idx1, idx2):
            atom = rwmol.GetAtomWithIdx(idx)
            atom.SetNoImplicit(False)

    elif edit_type == "reduce_multi_bond":
        pairs = candidate.get("target_pairs_in_pattern", info.get("target_pairs_in_pattern"))
        ring_atoms_pattern = candidate.get("ring_atoms_in_pattern", info.get("ring_atoms_in_pattern"))
        ring_bonds_pattern = candidate.get("ring_bonds_in_pattern", info.get("ring_bonds_in_pattern"))

        for pair in pairs:
            idx_c = match[pair[0]]
            idx_o = match[pair[1]]
            bond = rwmol.GetBondBetweenAtoms(idx_c, idx_o)
            if bond is None:
                return None
            bond.SetBondType(Chem.BondType.SINGLE)
            rwmol.GetAtomWithIdx(idx_o).SetNoImplicit(False)
            rwmol.GetAtomWithIdx(idx_c).SetNumExplicitHs(0)
            rwmol.GetAtomWithIdx(idx_c).SetNoImplicit(False)

        ring_indices = [match[i] for i in ring_atoms_pattern]
        for a in ring_indices:
            rwmol.GetAtomWithIdx(a).SetIsAromatic(True)

        for b1, b2 in ring_bonds_pattern:
            bidx1, bidx2 = match[b1], match[b2]
            rbond = rwmol.GetBondBetweenAtoms(bidx1, bidx2)
            if rbond is None:
                continue
            rbond.SetBondType(Chem.BondType.AROMATIC)
            rbond.SetIsAromatic(True)

    elif edit_type == "replace_multi":
        for sub in candidate["param"]:
            target_idx = match[sub["idx_in_pattern"]]
            atom = rwmol.GetAtomWithIdx(target_idx)
            atom.SetAtomicNum(sub["new_element"])
            atom.SetFormalCharge(sub.get("new_charge", 0))
            atom.SetNoImplicit(False)
            atom.SetNumExplicitHs(0)

    elif edit_type == "remove_substituent":
        remove_idx = match[candidate["remove_idx_in_pattern"]]
        upgrade_idx = match[candidate["upgrade_bond_to_idx_in_pattern"]]
        center_idx = match[candidate.get("center_idx_in_pattern", 0)]

        to_remove = set()
        visited = {center_idx}

        stack = [remove_idx]
        while stack:
            cur = stack.pop()
            if cur in visited:
                continue
            visited.add(cur)
            to_remove.add(cur)
            for n in mol.GetAtomWithIdx(cur).GetNeighbors():
                if n.GetIdx() not in visited:
                    stack.append(n.GetIdx())

        visited.add(upgrade_idx)
        upgrade_atom = mol.GetAtomWithIdx(upgrade_idx)
        for n in upgrade_atom.GetNeighbors():
            if n.GetIdx() != center_idx and n.GetIdx() not in to_remove:
                stack2 = [n.GetIdx()]
                while stack2:
                    cur2 = stack2.pop()
                    if cur2 in visited:
                        continue
                    visited.add(cur2)
                    to_remove.add(cur2)
                    for n2 in mol.GetAtomWithIdx(cur2).GetNeighbors():
                        if n2.GetIdx() not in visited:
                            stack2.append(n2.GetIdx())

        for ridx in sorted(to_remove, reverse=True):
            rwmol.RemoveAtom(ridx)

        def _adjust3(idx, removed):
            shift = sum(1 for r in removed if r < idx)
            return idx - shift

        center_new = _adjust3(center_idx, to_remove)
        upgrade_new = _adjust3(upgrade_idx, to_remove)

        bond = rwmol.GetBondBetweenAtoms(center_new, upgrade_new)
        if bond is None:
            return None
        bond.SetBondType(Chem.BondType.DOUBLE)
        rwmol.GetAtomWithIdx(center_new).SetNoImplicit(False)
        rwmol.GetAtomWithIdx(upgrade_new).SetNoImplicit(False)

    elif edit_type == "remove_atom":
        remove_idx = match[candidate["remove_idx_in_pattern"]]
        center_idx = match[candidate.get("center_idx_in_pattern", 0)]

        to_remove = set()
        visited = {center_idx}
        stack = [remove_idx]
        while stack:
            cur = stack.pop()
            if cur in visited:
                continue
            visited.add(cur)
            to_remove.add(cur)
            for n in mol.GetAtomWithIdx(cur).GetNeighbors():
                if n.GetIdx() not in visited:
                    stack.append(n.GetIdx())

        for ridx in sorted(to_remove, reverse=True):
            rwmol.RemoveAtom(ridx)

        def _adjust4(idx, removed):
            shift = sum(1 for r in removed if r < idx)
            return idx - shift

        center_new = _adjust4(center_idx, to_remove)
        rwmol.GetAtomWithIdx(center_new).SetNoImplicit(False)

    elif edit_type == "cleave_bond":
        pair = candidate["cleave_pair_in_pattern"]
        idx1 = match[pair[0]]
        idx2 = match[pair[1]]
        bond = rwmol.GetBondBetweenAtoms(idx1, idx2)
        if bond is None:
            return None
        rwmol.RemoveBond(idx1, idx2)
        for idx in (idx1, idx2):
            rwmol.GetAtomWithIdx(idx).SetNoImplicit(False)

    elif edit_type == "open_epoxide":
        pair = candidate["break_pair_in_pattern"]
        idx_o = match[pair[0]]
        idx_c_break = match[pair[1]]

        bond = rwmol.GetBondBetweenAtoms(idx_o, idx_c_break)
        if bond is None:
            return None
        rwmol.RemoveBond(idx_o, idx_c_break)

        frag = Chem.MolFromSmiles("O")
        if frag is None:
            return None
        combined = Chem.CombineMols(rwmol.GetMol(), frag)
        rwmol = Chem.RWMol(combined)
        offset = mol.GetNumAtoms()
        rwmol.AddBond(idx_c_break, offset, Chem.BondType.SINGLE)

        rwmol.GetAtomWithIdx(idx_o).SetNoImplicit(False)
        rwmol.GetAtomWithIdx(idx_c_break).SetNoImplicit(False)
        rwmol.GetAtomWithIdx(offset).SetNoImplicit(False)

    elif edit_type == "replace_ring":
        ring_key = candidate.get("ring_atom_indices_in_pattern", info.get("ring_atom_indices_in_pattern"))
        anchor_key = candidate.get("anchor_indices_in_pattern", info.get("anchor_indices_in_pattern"))
        ring_indices = [match[i] for i in ring_key]
        anchor_idx1 = match[anchor_key[0]]
        anchor_idx2 = match[anchor_key[1]]

        # 안전장치: 고리 원자가 anchor 2개 외에 다른 치환기(메틸기 등)를
        # 갖고 있으면, 그 치환기가 고아가 되어 분자가 조각나므로 치환을
        # 거부한다 (다중 BCP 치환 조각화 버그 재발 방지)
        ring_set = set(ring_indices)
        for ridx in ring_indices:
            ratom = mol.GetAtomWithIdx(ridx)
            for n in ratom.GetNeighbors():
                nidx = n.GetIdx()
                if nidx not in ring_set and nidx not in (anchor_idx1, anchor_idx2):
                    return None

        anchor1_ring_neighbor = None
        anchor2_ring_neighbor = None
        for ridx in ring_indices:
            ratom = mol.GetAtomWithIdx(ridx)
            neighbor_idxs = [n.GetIdx() for n in ratom.GetNeighbors()]
            if anchor_idx1 in neighbor_idxs:
                anchor1_ring_neighbor = ridx
            if anchor_idx2 in neighbor_idxs:
                anchor2_ring_neighbor = ridx

        if anchor1_ring_neighbor is None or anchor2_ring_neighbor is None:
            return None

        frag = Chem.MolFromSmiles(candidate["param"])
        if frag is None:
            return None

        for ridx in sorted(ring_indices, reverse=True):
            rwmol.RemoveAtom(ridx)

        def _adjust(idx, removed):
            shift = sum(1 for r in removed if r < idx)
            return idx - shift

        anchor_idx1_new = _adjust(anchor_idx1, ring_indices)
        anchor_idx2_new = _adjust(anchor_idx2, ring_indices)

        combined = Chem.CombineMols(rwmol.GetMol(), frag)
        rwmol2 = Chem.RWMol(combined)
        offset = rwmol.GetMol().GetNumAtoms()

        frag_attach1 = None
        frag_attach2 = None
        for atom in frag.GetAtoms():
            if atom.GetSymbol() == '*':
                map_num = atom.GetAtomMapNum()
                if map_num == 1:
                    frag_attach1 = atom.GetIdx() + offset
                elif map_num == 2:
                    frag_attach2 = atom.GetIdx() + offset

        if frag_attach1 is None or frag_attach2 is None:
            return None

        dummy1 = rwmol2.GetAtomWithIdx(frag_attach1)
        dummy2 = rwmol2.GetAtomWithIdx(frag_attach2)
        real_neighbor1 = dummy1.GetNeighbors()[0].GetIdx()
        real_neighbor2 = dummy2.GetNeighbors()[0].GetIdx()

        rwmol2.AddBond(anchor_idx1_new, real_neighbor1, Chem.BondType.SINGLE)
        rwmol2.AddBond(anchor_idx2_new, real_neighbor2, Chem.BondType.SINGLE)
        rwmol2.RemoveAtom(max(frag_attach1, frag_attach2))
        rwmol2.RemoveAtom(min(frag_attach1, frag_attach2))

        rwmol = rwmol2
    else:
        return None

    try:
        new_mol = rwmol.GetMol()
        Chem.SanitizeMol(new_mol)
    except Exception:
        return None

    new_smiles = Chem.MolToSmiles(new_mol)

    check_mol = Chem.MolFromSmiles(new_smiles)
    is_valid = check_mol is not None
    if is_valid:
        if edit_type != "cleave_bond" and '.' in new_smiles:
            is_valid = False
        for atom in check_mol.GetAtoms():
            if (atom.GetNoImplicit() and atom.GetFormalCharge() == 0
                    and atom.GetSymbol() in ('C', 'N', 'O')
                    and atom.GetTotalNumHs() == 0 and atom.GetDegree() < 4):
                is_valid = False
                break

    return {
        "new_smiles": new_smiles,
        "candidate_used": candidate["name"],
        "rationale": candidate["rationale"],
        "is_valid": is_valid,
    }


Overwriting src/tools/atom_editor.py


In [29]:
importlib.reload(src.tools.atom_editor)
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import propose_fix

print("안트라퀴논:", propose_fix("O=C1c2ccccc2C(=O)c2ccccc21", "quinone_A_anthraquinone", candidate_idx=0))
print("\n=== 회귀 테스트 ===")
print(propose_fix("O=C1C=CC(=O)C=C1", "quinone_A(370)", candidate_idx=0))

안트라퀴논: {'new_smiles': 'Oc1c2ccccc2c(O)c2ccccc12', 'candidate_used': 'anthrahydroquinone (reduced, re-aromatized)', 'rationale': '안트라퀴논은 벤조퀴논과 동일한 산화환원 사이클링(redox cycling) 메커니즘을 가지되, 두 벤젠 고리에 의해 안정화되어 항암제(독소루비신 등) 및 염료에서도 흔히 쓰이는 골격임. 두 카르보닐을 동시에 환원하고 중앙 고리를 재방향족화하여 안트라하이드로퀴논으로 전환, 산화환원 사이클링 능력을 제거함. 결과물이 hydroquinone 규칙에 해당할 수 있어 반복 루프가 자동으로 추가 개선 가능 (Murcko scaffold 분석으로 발견, 검증 필요)', 'is_valid': True}

=== 회귀 테스트 ===
{'new_smiles': 'Oc1ccc(O)cc1', 'candidate_used': 'hydroquinone (reduced, re-aromatized)', 'rationale': '파라벤조퀴논은 산화환원 사이클(redox cycling)을 통해 활성산소종(ROS)을 생성하고 DNA/단백질과 직접 공유결합하는 대표적 반응성 구조. 체내 NQO1(퀴논 환원효소) 효소가 실제로 수행하는 반응과 동일하게 두 카르보닐을 환원하고 고리를 재방향족화하여 안정적인 하이드로퀴논으로 전환. 결과물이 다시 hydroquinone 규칙에 해당할 수 있으며, 이 경우 반복 루프가 자동으로 메톡시페놀 등 산화에 더 안정적인 형태로 한 단계 더 개선함 (검증 필요, 안트라퀴논 등 융합고리형은 미지원)', 'is_valid': True}


In [30]:
!git add src/tools/replacement_library.py src/tools/atom_editor.py
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	modified:   src/tools/atom_editor.py
	modified:   src/tools/replacement_library.py

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	tox21_trainvalid.fragdb
	tox21_trainvalid.mmpdb
	tox21_trainvalid.smi
	tox21_trainvalid_filtered.fragdb
	tox21_trainvalid_filtered.mmpdb
	tox21_trainvalid_filtered.smi
	tox21_valid.fragdb
	tox21_valid.mmpdb
	tox21_valid.smi
	tox21_valid_properties.csv



In [32]:
with open(".gitignore", "a") as f:
    f.write("\n*.fragdb\n*.mmpdb\ntox21_*.smi\ntox21_*_properties.csv\n")

print("완료")
!tail -5 .gitignore

!git add .gitignore src/tools/atom_editor.py src/tools/replacement_library.py
!git status

완료

*.fragdb
*.mmpdb
tox21_*.smi
tox21_*_properties.csv
On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	modified:   .gitignore
	modified:   src/tools/atom_editor.py
	modified:   src/tools/replacement_library.py



In [33]:
!git add .gitignore src/tools/atom_editor.py src/tools/replacement_library.py
!git status
!git commit -m "Add quinone_A_anthraquinone and quinone_diimine rules (found via Murcko scaffold frequency comparison between toxicophore-positive and negative groups, per Tox21 guide methodology). Fix reduce_multi_bond: ring bond lookup failure was incorrectly returning None (treating a missing ring bond edge case as fatal) when RDKit's SanitizeMol can correctly aromatize with the remaining bond info alone; changed to skip missing bonds instead of aborting. Verified anthraquinone -> anthrahydroquinone success and quinone_A(370) regression unaffected. Library now 32 rules."
!git push origin main

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	modified:   .gitignore
	modified:   src/tools/atom_editor.py
	modified:   src/tools/replacement_library.py

[main 43d0bec] Add quinone_A_anthraquinone and quinone_diimine rules (found via Murcko scaffold frequency comparison between toxicophore-positive and negative groups, per Tox21 guide methodology). Fix reduce_multi_bond: ring bond lookup failure was incorrectly returning None (treating a missing ring bond edge case as fatal) when RDKit's SanitizeMol can correctly aromatize with the remaining bond info alone; changed to skip missing bonds instead of aborting. Verified anthraquinone -> anthrahydroquinone success and quinone_A(370) regression unaffected. Library now 32 rules.
 3 files changed, 38 insertions(+), 1 deletion(-)
Enumerating objects: 13, done.
Counting objects: 100% (13/13), done.
Delta compression using up to 2 threads
Compressing ob

In [34]:
import requests

base_url = "https://www.guidetopharmacology.org/services"

def search_ligand(name):
    resp = requests.get(f"{base_url}/ligands", params={"name": name})
    return resp.json()

def get_ligand_interactions(ligand_id):
    resp = requests.get(f"{base_url}/ligands/{ligand_id}/interactions")
    return resp.json()

dopamine_search = search_ligand("dopamine")
print(dopamine_search[:3] if dopamine_search else "결과 없음")

[{'ligandId': 13048, 'name': 'cerebral dopamine neurotrophic factor', 'type': 'Peptide', 'abbreviation': '', 'inn': '', 'approvalSource': '', 'approved': False, 'whoEssential': False, 'withdrawn': False, 'antibacterial': False, 'immuno': False, 'malaria': False, 'labelled': False, 'radioactive': False, 'activeDrugIds': [], 'prodrugIds': [], 'complexIds': [], 'subunitIds': [], 'species': 'Human'}, {'ligandId': 940, 'name': 'dopamine', 'type': 'Metabolite', 'abbreviation': '', 'inn': 'dopamine', 'approvalSource': 'FDA (1974)', 'approved': True, 'whoEssential': True, 'withdrawn': False, 'antibacterial': False, 'immuno': True, 'malaria': False, 'labelled': False, 'radioactive': False, 'activeDrugIds': [], 'prodrugIds': [], 'complexIds': [], 'subunitIds': []}, {'ligandId': 5552, 'name': 'N-oleoyldopamine', 'type': 'Metabolite', 'abbreviation': 'OLDA', 'inn': '', 'approvalSource': '', 'approved': False, 'whoEssential': False, 'withdrawn': False, 'antibacterial': False, 'immuno': False, 'mala

In [35]:
dopamine_interactions = get_ligand_interactions(940)
print(f"총 상호작용 수: {len(dopamine_interactions)}")

for interaction in dopamine_interactions[:15]:
    print(f"표적: {interaction.get('targetName')}, "
          f"타입: {interaction.get('type')}, "
          f"친화도지표: {interaction.get('affinityType')}, "
          f"값: {interaction.get('affinity')}, "
          f"단위: {interaction.get('affinityUnits', 'nM')}")

총 상호작용 수: 7
표적: D<sub>1</sub> receptor, 타입: Agonist, 친화도지표: None, 값: 4.3 - 5.6, 단위: nM
표적: D<sub>2</sub> receptor, 타입: Agonist, 친화도지표: None, 값: 4.7 - 7.2, 단위: nM
표적: D<sub>3</sub> receptor, 타입: Agonist, 친화도지표: None, 값: 6.4 - 7.3, 단위: nM
표적: D<sub>5</sub> receptor, 타입: Agonist, 친화도지표: None, 값: 6.6, 단위: nM
표적: D<sub>4</sub> receptor, 타입: Agonist, 친화도지표: None, 값: 7.6, 단위: nM
표적: D<sub>4</sub> receptor, 타입: Agonist, 친화도지표: None, 값: 7.4, 단위: nM
표적: D<sub>2</sub> receptor, 타입: Agonist, 친화도지표: None, 값: 5.3 - 6.4, 단위: nM


In [36]:
%%writefile src/tools/replacement_library.py

REPLACEMENT_LIBRARY = {
    "nitro_group": {
        "problem_smarts": "[N+](=O)[O-]",
        "candidates": [
            {"smiles": "N", "name": "primary amine",
             "rationale": "[참고] 메트로니다졸, 니트로푸란토인, 벤즈니다졸 등 일부 "
                          "항균제/항기생충제는 니트로기의 선택적 환원 활성화 자체가 "
                          "치료 메커니즘이므로, 이런 프로드러그 설계 맥락에서는 본 "
                          "치환이 적절하지 않을 수 있음. || 극성을 유지하면서 니트로기의 "
                          "환원성 대사 중간체 생성 경로를 제거함"},
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "약물유사 골격에서 흔히 쓰이는 안정적 대체기로, 수소결합 donor/acceptor 특성을 일부 유지"},
            {"smiles": "C#N", "name": "nitrile",
             "rationale": "대사 안정성이 개선된 사례가 문헌에 다수 보고됨, 다만 극성은 다소 감소"},
        ],
    },
    "aldehyde": {
        "problem_smarts": "[CX3H1](=O)",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "알데히드의 친전자성(단백질 부가물 형성 우려)을 제거하면서 유사한 형태 유지"},
            {"smiles": "C(O)", "name": "alcohol",
             "rationale": "가장 단순한 환원형 대체, 반응성 크게 감소"},
        ],
    },
    "Michael_acceptor_1": {
        "edit_method": "atom_edit",
        "problem_smarts": "C=CC(=O)",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "saturated (C-C single bond)",
             "rationale": "[참고] 에타크린산처럼 시스테인 잔기와의 공유결합 자체가 "
                          "작용 메커니즘인 공유결합 억제제(covalent inhibitor) "
                          "계열에는 본 경고가 그대로 적용되지 않을 수 있음. || "
                          "알파,베타-불포화 카르보닐의 C=C 이중결합을 환원하여 "
                          "단백질 친전자성 부가반응(Michael addition, covalent "
                          "binding) 위험을 제거함"},
        ],
    },
    "acid_halide": {
        "problem_smarts": "C(=O)[F,Cl,Br,I]",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "고반응성 아실할라이드를 안정적인 아마이드로 대체"},
            {"smiles": "C(=O)O", "name": "ester",
             "rationale": "아마이드보다 극성이 낮고 유연한 대체 옵션, 가수분해 속도 조절 가능 (검증 필요)"},
        ],
    },
    "alkyl_halide": {
        "problem_smarts": "[Cl,Br,I]",
        "candidates": [
            {"smiles": "O", "name": "hydroxyl (alcohol)",
             "rationale": "[참고] 메클로르에타민, 사이클로포스파미드, 카머스틴, "
                          "클로람부실 등 알킬화 항암제는 DNA 알킬화(반응성) 자체가 "
                          "세포독성 치료 메커니즘이므로, 이 계열에는 본 치환이 "
                          "적절하지 않음. || 이탈기를 제거해 알킬화 반응성을 없앰, "
                          "극성은 유사하게 유지"},
            {"smiles": "F", "name": "fluorine",
             "rationale": "할로겐을 유지하되 C-F 결합은 강해 이탈기로 작용하지 않음, 입체적 크기도 유사"},
        ],
    },
    "aniline": {
        "edit_method": "atom_edit",
        "problem_smarts": "[NH2]c1ccc([#6,#7,#8,#16])cc1",
        "target_idx_in_pattern": 0,
        "ring_atom_indices_in_pattern": [1, 2, 3, 4, 6, 7],
        "anchor_indices_in_pattern": (0, 5),
        "candidates": [
            {"edit_type": "add_substituent", "param": "C(=O)C",
             "target_idx_in_pattern": 0,
             "name": "acetamide (acylated amine)",
             "rationale": "[참고] 설파계 항생제(설파닐아마이드, 설파메톡사졸 등)와 "
                          "프로카인아마이드처럼 아닐린 골격이 반응성 대사가 아닌 "
                          "안정적 형태로 널리 처방되어 온 사례가 다수 있음. 이 경우 "
                          "특이체질 반응은 드물고 예측이 어려워, 본 경고를 절대적 "
                          "배제 기준이 아닌 참고 신호로 해석해야 함. || 1차 방향족 "
                          "아민을 아마이드로 아실화하여 N-hydroxylation 경로 자체를 차단"},
            {"edit_type": "replace_ring", "param": "[*:1]C12CC(C1)(C2)[*:2]",
             "ring_atom_indices_in_pattern": [1, 2, 3, 4, 6, 7],
             "anchor_indices_in_pattern": (0, 5),
             "name": "BCP (bicyclo[1.1.1]pentane)",
             "rationale": "para-이치환 아닐린의 방향족 벤젠 고리를 포화 bicyclic "
                          "탄소골격(BCP)으로 교체함. 방향족성 제거로 aniline reactive "
                          "metabolite(RM) 형성 및 CYP-inhibition을 감소시켜, 퀴논이민 "
                          "생성 경로를 차단하고 특이체질 약물 부작용(IADR) 위험을 낮춤 "
                          "(문헌 근거, 학생 제공). 벤젠과의 공간적 유사성, Fsp3 증가, "
                          "실제 성공 사례가 많아 채택. 아마이드화(단순 아민 치환)보다 "
                          "변화 폭이 크지만, 물성 개선 효과도 더 큼"},
        ],
    },
    "Sulfonic_acid_2": {
        "problem_smarts": "[#6]S(=O)(=O)[OX2H1,OX1-]",
        "candidates": [
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "[참고] 암페타민 설페이트, 사퀴나비르 메실레이트처럼 "
                          "일부 승인약물에서 설폰산/설폰산 유사기는 활성 골격이 "
                          "아니라 염(salt) 형성을 위한 카운터이온으로만 존재함. "
                          "이 경우 본 규칙이 다루는 '독성 유발 골격'과 무관하므로, "
                          "치환 대상 여부를 판단하기 전에 이 산이 활성 골격의 "
                          "일부인지 염 형성용인지 구분이 필요함. || 생리적 pH에서 "
                          "이온화 정도(전하)를 크게 낮춰 세포막 투과성을 개선함. "
                          "설폰산은 대부분 음이온 상태로 존재해 경구 흡수가 저해되는 "
                          "경우가 많으나, 설폰아마이드는 유사한 골격을 유지하면서도 "
                          "중성에 가까워 약물유사성이 개선됨"},
            {"smiles": "C(=O)O", "name": "carboxylic acid",
             "rationale": "설폰산보다 산성도가 약하고 부피가 작은 산성 bioisostere "
                          "(검증 필요)"},
        ],
    },
    "imine_1_oxime": {
        "edit_method": "atom_edit",
        "problem_smarts": "C=N[OX2H1]",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "amine (reduced)",
             "rationale": "옥심의 C=N 결합을 환원하여, 가수분해 시 원래의 반응성 "
                          "카르보닐(알데히드/케톤)로 되돌아갈 수 있는 대사 불안정 "
                          "경로를 제거함"},
        ],
    },
    "imine_1_general": {
        "edit_method": "atom_edit",
        "problem_smarts": "[CX3;!$(C(N)(N)=N)]=N",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "amine (reduced)",
             "rationale": "일반 이민(C=N-R)을 환원하여 가수분해 시 반응성 카르보닐로 "
                          "되돌아갈 수 있는 대사 불안정 경로를 제거함. 옥심 특유의 "
                          "메커니즘보다는 근거가 다소 약하며, 하위 구조별 개별 검증 필요. "
                          "구아니딘(N-C(=N)-N, 공명구조로 일반 이민과 반응성이 다름)은 "
                          "이 SMARTS에서 명시적으로 제외함"},
        ],
    },
    "catechol": {
        "edit_method": "atom_edit",
        "problem_smarts": "[OX2H;$(Oc1ccccc1O)]",
        "target_idx_in_pattern": 0,
        "candidates": [
            {"edit_type": "add_substituent", "param": "C", "name": "methoxy",
             "rationale": "[참고] 도파민, 에피네프린, 이소프로테레놀 등 카테콜아민류 "
                      "약물은 카테콜 구조 자체가 아드레날린/도파민 수용체 결합에 "
                      "필수적인 약효 골격이므로, 이 경우 본 치환은 독성 감소가 "
                      "아니라 약효 상실로 이어짐. 실제 도파민은 도파민 수용체 "
                      "D1(Ki 4.3-5.6 nM), D2(Ki 4.7-7.2 nM), D3(Ki 6.4-7.3 nM)에 "
                      "단자릿수 나노몰 수준의 강력한 작용제 친화도를 가짐(IUPHAR/BPS "
                      "Guide to PHARMACOLOGY 확인). || 인체의 COMT(catechol-O-"
                      "methyltransferase) 효소가 카테콜을 메톡시페놀로 메틸화하여 "
                      "해독하는 생리적 경로와 동일한 원리. 오르토-퀴논으로의 산화 "
                      "경로를 차단하여 세포독성/유전독성 우려를 낮춤 (학생 확인 "
                      "예정: ScienceDirect catechol overview, PMC6643002 등 참고)"},
         ],
    },
    "Thiocarbonyl_group": {
        "edit_method": "atom_edit",
        "problem_smarts": "[#6]=[#16]",
        "target_idx_in_pattern": 1,
        "candidates": [
            {"edit_type": "replace_element", "param": 8, "name": "carbonyl (O replacing S)",
             "rationale": "[참고] 티오펜탈·티아밀랄(치오바르비투레이트, C=S가 지용성 "
                          "증가로 빠른 마취효과에 기여)과 티오구아닌(퓨린 유사 항대사물, "
                          "황이 작용기전에 필수)처럼 황 원자가 약효/효력에 직접 "
                          "기여하는 경우가 있어, 이 계열에는 본 치환이 부적절할 수 "
                          "있음. || 황을 산소로 대체(티오카르보닐->카르보닐)하는 것은 "
                          "흔한 bioisostere 전략으로, 갑상선 기능 저해 등 황 함유 "
                          "작용기 특유의 대사/독성 우려를 낮춤 (검증 필요, "
                          "thiourea->urea 치환 논리와 동일 계열)"},
        ],
    },
    "thiol_2": {
        "problem_smarts": "[SX2H1]",
        "candidates": [
            {"smiles": "O", "name": "hydroxyl (alcohol)",
             "rationale": "티올의 금속 킬레이팅 및 산화(이황화물/술펜산 형성) 반응성을 "
                          "제거하면서, 극성·수소결합 특성을 유사하게 유지함"},
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "티올을 아마이드로 대체하여 반응성을 낮추면서 약물유사 골격에서 "
                          "흔히 쓰이는 안정적 작용기로 전환 (검증 필요)"},
        ],
    },
    "thiol_1_dithiocarbamate": {
        "edit_method": "atom_edit",
        "problem_smarts": "C(=S)[SX1-]",
        "candidates": [
            {"edit_type": "replace_multi",
             "param": [
                 {"idx_in_pattern": 1, "new_element": 8, "new_charge": 0},
                 {"idx_in_pattern": 2, "new_element": 7, "new_charge": 0},
             ],
             "name": "carbamate (O,N replacing S,S)",
             "rationale": "디티오카바메이트(R-O-C(=S)-S-)를 카바메이트(R-O-C(=O)-N)로 "
                          "전환. 두 황 원자를 각각 산소·질소로 교체하여 금속 킬레이팅 "
                          "능력과 효소 억제 활성(디티오카바메이트류 특유의 살충제성 "
                          "독성 기전)을 제거함 (검증 필요)"},
        ],
    },
    "thiol_1_thiocarboxylate": {
        "edit_method": "atom_edit",
        "problem_smarts": "[SX1-]C(=O)",
        "target_idx_in_pattern": 0,
        "candidates": [
            {"edit_type": "replace_element", "param": 8, "name": "carboxylate (O replacing S)",
             "rationale": "티오카르복실산 음이온(R-C(=O)-S-)의 황을 산소로 대체하여 "
                          "카르복실산염(R-C(=O)-O-)으로 전환. 황 원자의 금속 킬레이팅 "
                          "및 친핵성 반응성을 제거함 (검증 필요)"},
        ],
    },
    "het-C-het_not_in_ring": {
        "edit_method": "atom_edit",
        "problem_smarts": "[CX4](O)(O)",
        "candidates": [
            {"edit_type": "remove_substituent",
             "center_idx_in_pattern": 0,
             "remove_idx_in_pattern": 1,
             "upgrade_bond_to_idx_in_pattern": 2,
             "name": "ketone/ester (one alkoxy removed, C=O formed)",
             "rationale": "아세탈/케탈 또는 오르토에스터(탄소 하나에 알콕시기 2개 "
                          "이상)는 가수분해에 민감하여 반응성 카르보닐(케톤/알데히드)로 "
                          "쉽게 분해되며 대사 불안정성을 일으킴. 알콕시기 하나를 제거하고 "
                          "남은 산소를 카르보닐로 승격시켜, 가수분해로 어차피 도달할 "
                          "안정한 최종 형태로 미리 전환함 (검증 필요)"},
        ],
    },
    "hydroquinone": {
        "edit_method": "atom_edit",
        "problem_smarts": "[OX2H]c1ccc([OX2H,NX3H1,NX3H2])cc1",
        "target_idx_in_pattern": 0,
        "candidates": [
            {"edit_type": "add_substituent", "param": "C", "name": "methoxy",
             "rationale": "[참고] 아세트아미노펜은 정상 용량에서는 안전하며 과다복용 "
                          "시에만 위험한 용량 의존적 사례임. 본 시스템은 치료지수를 "
                          "고려하지 않으므로, 아트로핀·디곡신·와파린처럼 좁은 치료지수를 "
                          "가진 기존 약물 전반에 유사하게 적용되는 한계임. || 파라 "
                          "위치에 OH와 (OH 또는 NH)가 있는 구조(하이드로퀴논/파라-"
                          "아미노페놀 계열)는 산화되어 파라-퀴논 또는 파라-퀴논이민(예: "
                          "아세트아미노펜의 NAPQI)을 형성, 글루타치온 고갈과 단백질 "
                          "공유결합을 통한 간독성 위험이 있음"},
        ],
    },
    "azo_A(324)": {
        "edit_method": "atom_edit",
        "problem_smarts": "N=N",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "hydrazine (reduced)",
             "rationale": "아조기(N=N)는 체내에서 아조환원효소에 의해 환원되어 두 개의 "
                          "방향족 아민으로 분해되며, 그 중 일부(벤지딘류 등)가 발암성을 "
                          "가지는 것으로 잘 알려짐(아조 색소의 대표적 독성 메커니즘). "
                          "이중결합을 환원하여 하이드라진 형태로 전환, 완전한 아민 "
                          "분해 경로 자체를 차단함 (검증 필요: 하이드라진 자체의 "
                          "잔여 반응성은 추가 확인 필요)"},
        ],
    },
    "Three-membered_heterocycle": {
        "edit_method": "atom_edit",
        "problem_smarts": "[CX4]1[OX2][CX4]1",
        "candidates": [
            {"edit_type": "open_epoxide", "break_pair_in_pattern": (1, 2),
             "name": "vicinal diol (ring-opened)",
             "rationale": "에폭시드(3원자 고리, 옥시란)는 고리 변형(strain)으로 인해 "
                          "친핵체(DNA, 단백질)와 쉽게 반응하는 알킬화제로 작용함. "
                          "체내 에폭시드 가수분해효소(epoxide hydrolase)가 실제로 "
                          "수행하는 반응과 동일하게 고리를 열어 비시날 디올(vicinal "
                          "diol)로 전환, 반응성을 제거함"},
        ],
    },
    "diketo_group": {
        "edit_method": "atom_edit",
        "problem_smarts": "C(=O)C(=O)",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "alpha-hydroxy ketone (reduced)",
             "rationale": "비시날 알파-디케톤(1,2-diketone)은 반응성이 높은 친전자체로 "
                          "단백질과 부가물을 형성할 수 있으며, 흡입 시 호흡기 독성을 "
                          "일으키는 것으로 알려진 디아세틸(버터향 첨가제) 사례가 대표적임. "
                          "카르보닐 하나를 환원하여 알파-하이드록시케톤(아실로인)으로 "
                          "전환, 케토-환원효소에 의한 실제 해독 경로와 유사한 방향으로 "
                          "반응성을 낮춤 (검증 필요)"},
        ],
    },
    "thioester": {
        "edit_method": "atom_edit",
        "problem_smarts": "[SX2](C(=O))",
        "target_idx_in_pattern": 0,
        "candidates": [
            {"edit_type": "replace_element", "param": 8, "name": "ester (O replacing S)",
             "rationale": "티오에스터의 황을 산소로 대체하여 일반 에스터로 전환. "
                          "티오에스터는 일반 에스터보다 가수분해 반응성이 높고 아실화 "
                          "능력이 강해 단백질 등과 부반응 우려가 있음 (검증 필요)"},
        ],
    },
    "N-nitroso": {
        "edit_method": "atom_edit",
        "problem_smarts": "[NX2;+0;!$(N(=O)[O-])]=[OX1;+0]",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "N-hydroxylamine (reduced)",
             "rationale": "N-니트로소 화합물(니트로사민)은 대사 활성화(알파-수산화)를 "
                          "거쳐 강력한 알킬화 발암물질을 생성하는 것으로 잘 알려짐 "
                          "(발사르탄, 라니티딘 등 실제 의약품 불순물 리콜 사례). "
                          "N=O를 환원하여 반응성을 낮춤 (검증 필요: 완전한 해독은 "
                          "탈니트로소화가 필요하며 이는 근사적 접근)"},
        ],
    },
    "hydrazine": {
        "edit_method": "atom_edit",
        "problem_smarts": "[NX3H2][NX3H1]",
        "center_idx_in_pattern": 1,
        "candidates": [
            {"edit_type": "remove_atom",
             "remove_idx_in_pattern": 0,
             "center_idx_in_pattern": 1,
             "name": "amide/amine (terminal N removed)",
             "rationale": "하이드라진/하이드라지드(R-NH-NH2)의 말단 질소를 제거하여 "
                          "단순 아민 또는 아마이드로 되돌림. 하이드라진류는 대사 시 "
                          "반응성 디아제늄 중간체를 형성해 유전독성을 일으킬 수 있는 "
                          "것으로 알려짐. 이는 azo_A(324) 환원 시 생성되는 하이드라진 "
                          "중간체의 잔여 위험을 추가로 낮추는 후속 규칙이기도 함 "
                          "(검증 필요)"},
        ],
    },
    "sulphate": {
        "edit_method": "atom_edit",
        "problem_smarts": "[OX2][SX4](=O)(=O)[OX1,OX2H]",
        "center_idx_in_pattern": 0,
        "candidates": [
            {"edit_type": "remove_atom",
             "remove_idx_in_pattern": 1,
             "center_idx_in_pattern": 0,
             "name": "alcohol (sulfate group removed)",
             "rationale": "알킬 설페이트 에스터(R-O-SO3-)는 대사되어 반응성 있는 "
                          "설페이트 이탈기를 통한 알킬화제로 작용할 수 있음(디메틸설페이트가 "
                          "강력한 발암/독성 물질로 잘 알려진 대표 사례). 설페이트기 전체를 "
                          "제거하여 원래의 알코올로 되돌림 (검증 필요)"},
        ],
    },
    "N_oxide": {
        "edit_method": "atom_edit",
        "problem_smarts": "[n+][O-]",
        "center_idx_in_pattern": 0,
        "candidates": [
            {"edit_type": "remove_atom",
             "remove_idx_in_pattern": 1,
             "center_idx_in_pattern": 0,
             "name": "pyridine (N-oxide removed)",
             "rationale": "방향족 N-옥사이드는 산화적 대사산물이자 반응성 중간체 "
                          "생성 경로의 일부일 수 있음. 산소를 제거하여 원래의 중성 "
                          "방향족 아민(피리딘 등)으로 환원, 자연 대사에서의 환원 "
                          "경로와 유사한 방향으로 반응성을 낮춤 (검증 필요)"},
        ],
    },
    "2-halo_pyridine": {
        "edit_method": "atom_edit",
        "problem_smarts": "n:c(-[Cl,Br,I])",
        "center_idx_in_pattern": 1,
        "candidates": [
            {"edit_type": "remove_atom",
             "remove_idx_in_pattern": 2,
             "center_idx_in_pattern": 1,
             "name": "pyridine (halogen removed)",
             "rationale": "피리딘 고리 질소에 인접한 위치의 할로겐(특히 불소/염소)은 "
                          "친핵성 방향족 치환(SNAr) 반응에 취약해, 체내 친핵체(글루타치온, "
                          "단백질 시스테인 등)와 반응할 수 있음. 할로겐을 제거하고 수소로 "
                          "대체하여 이 반응성 경로를 차단함 (검증 필요)"},
        ],
    },
    "disulphide": {
        "edit_method": "atom_edit",
        "problem_smarts": "[SX2][SX2]",
        "candidates": [
            {"edit_type": "cleave_bond", "cleave_pair_in_pattern": (0, 1),
             "name": "two thiols (bond cleaved)",
             "rationale": "[참고] 이황화결합(S-S)은 시스틴/단백질의 3차구조 형성에 "
                          "필수적인 정상 생체 구조이기도 하므로, 이 결합이 약물의 "
                          "구조 안정성이나 표적 결합에 관여하는 경우 본 치환이 "
                          "부적절할 수 있음. || 디티오카바메이트류(티우람 등) 농약/"
                          "살균제에서 흔한 반응성 이황화결합을 두 개의 티올로 분리, "
                          "산화·금속킬레이팅 반응성을 낮춤 (검증 필요)"},
        ],
    },
    "quinone_A(370)": {
        "edit_method": "atom_edit",
        "problem_smarts": "O=C1C=CC(=O)C=C1",
        "target_pairs_in_pattern": [(1, 0), (4, 5)],
        "ring_atoms_in_pattern": [1, 2, 3, 4, 6, 7],
        "ring_bonds_in_pattern": [(1, 2), (2, 3), (3, 4), (4, 6), (6, 7), (7, 1)],
        "candidates": [
            {"edit_type": "reduce_multi_bond", "name": "hydroquinone (reduced, re-aromatized)",
             "rationale": "파라벤조퀴논은 산화환원 사이클(redox cycling)을 통해 활성산소종(ROS)을 "
                          "생성하고 DNA/단백질과 직접 공유결합하는 대표적 반응성 구조. 체내 "
                          "NQO1(퀴논 환원효소) 효소가 실제로 수행하는 반응과 동일하게 두 카르보닐을 "
                          "환원하고 고리를 재방향족화하여 안정적인 하이드로퀴논으로 전환. 결과물이 "
                          "다시 hydroquinone 규칙에 해당할 수 있으며, 이 경우 반복 루프가 자동으로 "
                          "메톡시페놀 등 산화에 더 안정적인 형태로 한 단계 더 개선함 (검증 필요, "
                          "안트라퀴논 등 융합고리형은 미지원)"},
        ],
    },
    "quinone_A_anthraquinone": {
        "edit_method": "atom_edit",
        "problem_smarts": "O=C1c2ccccc2C(=O)c2ccccc21",
        "target_pairs_in_pattern": [(1, 0), (8, 9)],
        "ring_atoms_in_pattern": [1, 2, 3, 4, 5, 6, 7, 8],
        "ring_bonds_in_pattern": [(1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7), (7, 8), (8, 1)],
        "candidates": [
            {"edit_type": "reduce_multi_bond", "name": "anthrahydroquinone (reduced, re-aromatized)",
             "rationale": "안트라퀴논은 벤조퀴논과 동일한 산화환원 사이클링(redox cycling) 메커니즘을 "
                          "가지되, 두 벤젠 고리에 의해 안정화되어 항암제(독소루비신 등) 및 염료에서도 "
                          "흔히 쓰이는 골격임. 두 카르보닐을 동시에 환원하고 중앙 고리를 재방향족화하여 "
                          "안트라하이드로퀴논으로 전환, 산화환원 사이클링 능력을 제거함. 결과물이 "
                          "hydroquinone 규칙에 해당할 수 있어 반복 루프가 자동으로 추가 개선 가능 "
                          "(Murcko scaffold 분석으로 발견, 검증 필요)"},
        ],
    },
    "quinone_diimine": {
        "edit_method": "atom_edit",
        "problem_smarts": "N=C1C=CC(=N)C=C1",
        "target_pairs_in_pattern": [(1, 0), (4, 5)],
        "ring_atoms_in_pattern": [1, 2, 3, 4, 6, 7],
        "ring_bonds_in_pattern": [(1, 2), (2, 3), (3, 4), (4, 6), (6, 7), (7, 1)],
        "candidates": [
            {"edit_type": "reduce_multi_bond", "name": "phenylenediamine (reduced, re-aromatized)",
             "rationale": "퀴논디이민(quinone diimine)은 벤조퀴논의 산소가 이민으로 치환된 유사체로, "
                          "동일한 산화환원 사이클링 메커니즘을 가지며 헤어염료 성분(파라페닐렌디아민 "
                          "산화형) 등에서 피부 알레르기 및 접촉성 피부염을 유발하는 것으로 알려짐. 두 "
                          "이민을 동시에 환원하고 고리를 재방향족화하여 페닐렌디아민(원래의 안정한 "
                          "환원형)으로 전환 (Murcko scaffold 분석으로 발견, 검증 필요)"},
        ],
    },
    "isocyanate": {
        "edit_method": "atom_edit",
        "problem_smarts": "[NX2]=[CX2]=[OX1]",
        "center_idx_in_pattern": 0,
        "candidates": [
            {"edit_type": "remove_atom",
             "remove_idx_in_pattern": 1,
             "center_idx_in_pattern": 0,
             "name": "amine (NCO hydrolyzed)",
             "rationale": "이소시아네이트(R-N=C=O)는 매우 반응성이 높은 친전자체로, "
                          "단백질/아미노기와 쉽게 부가반응을 일으켜 직업성 천식·과민증을 "
                          "유발하는 것으로 잘 알려짐(TDI, MDI 등 산업용 이소시아네이트 "
                          "사례). 체내/환경에서 실제로 일어나는 가수분해 경로(R-NCO + H2O "
                          "-> R-NH2 + CO2)와 동일하게 카르보닐 탄소와 산소를 제거하고 "
                          "질소만 남겨 아민으로 전환 (검증 필요)"},
        ],
    },
    "triple_bond": {
        "problem_smarts": "C#C",
        "edit_method": "atom_edit",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "alkene (partially reduced)",
             "rationale": "말단 알카인(삼중결합)은 CYP450 효소에 의해 기계기반 억제"
                          "(mechanism-based inhibition) 경로로 대사되며, 반응성 케텐/"
                          "에폭사이드 중간체를 형성해 효소를 비가역적으로 불활성화할 "
                          "수 있음(에티닐에스트라디올 등에서 알려진 메커니즘). 삼중결합을 "
                          "이중결합으로 환원하여 반응성을 낮춤 (검증 필요, 완전 포화가 "
                          "아닌 부분 환원)"},
        ],
    },
    "stilbene": {
        "problem_smarts": "c-[CX3]=[CX3]-c",
        "edit_method": "atom_edit",
        "target_idx_pair_in_pattern": (1, 2),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "diarylethane (reduced)",
             "rationale": "스틸벤 구조(두 방향족 고리를 잇는 C=C)는 디에틸스틸베스트롤"
                          "(DES)처럼 내분비교란 및 대사 산화를 통한 반응성 중간체 형성이 "
                          "알려진 골격. 이중결합을 환원하여 평면성을 낮추고 대사 반응성을 "
                          "완화함 (검증 필요, 에스트로겐 수용체 결합에 필요한 형태 자체를 "
                          "훼손할 수 있어 신중한 해석 필요)"},
        ],
    },
    "beta-keto/anhydride": {
        "edit_method": "atom_edit",
        "problem_smarts": "C(=O)OC(=O)",
        "center_idx_in_pattern": 2,
        "candidates": [
            {"edit_type": "remove_atom",
             "remove_idx_in_pattern": 3,
             "center_idx_in_pattern": 2,
             "name": "carboxylic acid (anhydride hydrolyzed)",
             "rationale": "산 무수물(R-C(=O)-O-C(=O)-R')은 강한 아실화제로 단백질 아미노산 "
                          "잔기와 쉽게 반응하며, 수용액 환경에서 자발적으로 가수분해되어 "
                          "두 개의 카르복실산으로 분해되는 것이 자연스러운 무독화 경로임. "
                          "한쪽 아실기를 제거하여 이 가수분해 최종형(카르복실산)으로 직접 "
                          "전환 (검증 필요). ※ 대안 후보(무수물->아마이드/이미드 bioisostere) "
                          "는 문헌 확인 후 추가 예정"},
        ],
    },
}

def get_replacement_candidates(rule_name: str) -> dict | None:
    """rule_name에 해당하는 치환 정보(SMARTS + 후보 리스트)를 반환. 없으면 None."""
    return REPLACEMENT_LIBRARY.get(rule_name)


Overwriting src/tools/replacement_library.py


In [37]:
importlib.reload(src.tools.replacement_library)
result_check = get_replacement_candidates("catechol")
print(result_check['candidates'][0]['rationale'])

[참고] 도파민, 에피네프린, 이소프로테레놀 등 카테콜아민류 약물은 카테콜 구조 자체가 아드레날린/도파민 수용체 결합에 필수적인 약효 골격이므로, 이 경우 본 치환은 독성 감소가 아니라 약효 상실로 이어짐. 실제 도파민은 도파민 수용체 D1(Ki 4.3-5.6 nM), D2(Ki 4.7-7.2 nM), D3(Ki 6.4-7.3 nM)에 단자릿수 나노몰 수준의 강력한 작용제 친화도를 가짐(IUPHAR/BPS Guide to PHARMACOLOGY 확인). || 인체의 COMT(catechol-O-methyltransferase) 효소가 카테콜을 메톡시페놀로 메틸화하여 해독하는 생리적 경로와 동일한 원리. 오르토-퀴논으로의 산화 경로를 차단하여 세포독성/유전독성 우려를 낮춤 (학생 확인 예정: ScienceDirect catechol overview, PMC6643002 등 참고)


In [38]:
etacrynic_search = search_ligand("etacrynic acid")
print(etacrynic_search[:3] if etacrynic_search else "결과 없음")

[{'ligandId': 7179, 'name': 'ethacrynic acid', 'type': 'Synthetic organic', 'abbreviation': '', 'inn': 'etacrynic acid', 'approvalSource': 'FDA (1967)', 'approved': True, 'whoEssential': False, 'withdrawn': False, 'antibacterial': False, 'immuno': False, 'malaria': False, 'labelled': False, 'radioactive': False, 'activeDrugIds': [], 'prodrugIds': [], 'complexIds': [], 'subunitIds': []}]


In [39]:
ethacrynic_interactions = get_ligand_interactions(7179)
print(f"총 상호작용 수: {len(ethacrynic_interactions)}")

for interaction in ethacrynic_interactions[:15]:
    print(f"표적: {interaction.get('targetName')}, "
          f"타입: {interaction.get('type')}, "
          f"값: {interaction.get('affinity')}, "
          f"단위: {interaction.get('affinityUnits', 'nM')}")

총 상호작용 수: 0


In [40]:
etacrynic_search2 = search_ligand("etacrynic")
print(etacrynic_search2)

[{'ligandId': 7179, 'name': 'ethacrynic acid', 'type': 'Synthetic organic', 'abbreviation': '', 'inn': 'etacrynic acid', 'approvalSource': 'FDA (1967)', 'approved': True, 'whoEssential': False, 'withdrawn': False, 'antibacterial': False, 'immuno': False, 'malaria': False, 'labelled': False, 'radioactive': False, 'activeDrugIds': [], 'prodrugIds': [], 'complexIds': [], 'subunitIds': []}]


In [41]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   src/tools/replacement_library.py

no changes added to commit (use "git add" and/or "git commit -a")


In [42]:
!git add src/tools/replacement_library.py

In [43]:
!git status
!git commit -m "Session 30 wrap-up: add quinone_A_anthraquinone and quinone_diimine rules (discovered via Murcko scaffold frequency comparison between toxicophore-positive/negative groups per Tox21 guide methodology), enhance catechol rationale with quantitative GtoPdb data (dopamine D1/D2/D3 receptor Ki values, single-digit nM). Library now 32 rules. MolVS standardization evaluated and rejected (Uncharger conflicts with rules requiring explicit ionization state, e.g. thiol_1_dithiocarbamate, Sulfonic_acid_2) - documented in limitations.md."
!git push origin main

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	modified:   src/tools/replacement_library.py

[main 2e16453] Session 30 wrap-up: add quinone_A_anthraquinone and quinone_diimine rules (discovered via Murcko scaffold frequency comparison between toxicophore-positive/negative groups per Tox21 guide methodology), enhance catechol rationale with quantitative GtoPdb data (dopamine D1/D2/D3 receptor Ki values, single-digit nM). Library now 32 rules. MolVS standardization evaluated and rejected (Uncharger conflicts with rules requiring explicit ionization state, e.g. thiol_1_dithiocarbamate, Sulfonic_acid_2) - documented in limitations.md.
 1 file changed, 11 insertions(+), 8 deletions(-)
Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 1014 bytes | 1014.00 KiB/s, done.
Total 5 (d